In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import math
import random
from typing import Tuple, List, Dict
import copy
import time
import psutil

import inspect


import h5py
import joblib


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils import weight_norm

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report

import tensorflow as tf


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

## Configurations

In [ ]:
file_path = f'drive/MyDrive/FYP- DATA/csv_labeled_data/combined_matches.csv'
# file_path = f'/content/drive/My Drive/Fyp/csv_labeled_data/match_{match_number}/labeled_data_match_{match_number}.csv'

USE_PREPROCESSED = False
PREPROCESSED_DIR = '/kaggle/input/preprocessed-sequences/'  # Change path if needed for Kaggle: '/kaggle/input/preprocessed-sequences/'



N_ROWS = 8
N_COLS = 18
NUM_ZONES = N_ROWS * N_COLS

# Sequence & horizon
SEQ_LEN =100        # number of past frames used
FPS = 25            # adjust based on your data (often 25fps for football)
HORIZON_SECONDS =3 # predict location after 3/5 seconds
HORIZON_FRAMES = HORIZON_SECONDS * FPS

# Training params
BATCH_SIZE =32
LR = 5e-4
EPOCHS = 1
PATIENCE = 2  # for early stopping
CO_ORDINATES= False
# Enhanced features for better zone prediction
FEATURE_COLS = [
    # Position
    "x_normalized", "y_normalized",

    # Basic movement
    "dx", "dy", "speed_normalized", "acceleration", "movement_angle",

    # # Multi-window velocities (NEW)
    # "dx_avg_3", "dy_avg_3", "dx_avg_5", "dy_avg_5", "dx_avg_10", "dy_avg_10",
    # "speed_avg_3", "speed_avg_5", "speed_avg_10",

    # Movement trends (NEW)
    "acceleration_trend", "angle_change", "angle_stability", "speed_change_rate",

    # Spatial context
    "distance_from_center", "distance_from_goal_home", "distance_from_goal_away",
    "distance_from_sideline",

    # Ball & team
    "distance_to_ball", "ball_possession_proximity", "team_has_possession", "team_spread",

    'home_score', 'away_score','nearest_opponent_1', 'nearest_opponent_2', 'nearest_opponent_3', 'defensive_line_distance','team_centroid_distance'


]


coord_cols=("x_normalized", "y_normalized")

some_number= 42

worker_num=4

MODEL_TYPE="Keras_tcn" #tcn/enhanced/tcn_new/LocusLab_tcn/Keras_tcn
BACKEND = "keras" #keras/torch

print(f"Using {len(FEATURE_COLS)} features for prediction")
print(f"Grid configuration: {N_ROWS}x{N_COLS} = {NUM_ZONES} zones")
print(f"Prediction horizon: {HORIZON_SECONDS} seconds ({HORIZON_FRAMES} frames)")
print(f"Using framework:{BACKEND}")
print(f"Using Model:{MODEL_TYPE}")
print(f"Using framework:{BACKEND}")
print(f"Using batch size: {BATCH_SIZE}")


In [ ]:
def xy_to_zone_vectorized(xs, ys, n_rows=N_ROWS, n_cols=N_COLS, flip_y=False):
    xs = np.asarray(xs)
    ys = np.asarray(ys)

    if flip_y:
        ys = 1 - ys

    row = np.clip((ys * n_rows).astype(int), 0, n_rows - 1)
    col = np.clip((xs * n_cols).astype(int), 0, n_cols - 1)
    return row * n_cols + col


In [ ]:
def split_by_match_df(df, train_frac=0.7, val_frac=0.15, seed=42, fixed_split=False):
    """
    Splits a dataframe into train/val/test by unique match_id.
    Ensures no leakage across matches.

    If fixed_split=True:
        3 matches -> train
        1 match   -> val
        1 match   -> test
    """
    unique_matches = df["match_id"].unique()
    np.random.seed(seed)
    np.random.shuffle(unique_matches)

    n_total = len(unique_matches)

    if fixed_split:
        if n_total < 5:
            raise ValueError("Need at least 5 unique matches for fixed split (3/1/1).")

        train_matches = unique_matches[:4]
        val_matches   = unique_matches[4:5]
        test_matches  = unique_matches[5:7]

    else:
        n_train = max(1, int(train_frac * n_total))
        n_val   = max(1, int(val_frac * n_total))
        n_train = min(n_train, n_total - n_val)

        train_matches = unique_matches[:n_train]
        val_matches   = unique_matches[n_train:n_train + n_val]
        test_matches  = unique_matches[n_train + n_val:]

    train_df = df[df["match_id"].isin(train_matches)].reset_index(drop=True)
    val_df   = df[df["match_id"].isin(val_matches)].reset_index(drop=True)
    test_df  = df[df["match_id"].isin(test_matches)].reset_index(drop=True)

    print("Split matches:", len(train_matches), len(val_matches), len(test_matches))
    print("Rows per split:", len(train_df), len(val_df), len(test_df))

    return train_df, val_df, test_df


## Football Sequence Dataset class & pytorch

In [ ]:
class FootballSequenceDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, horizon=HORIZON_FRAMES,
                 flip_y=False, feature_cols=FEATURE_COLS, scaler=None):
        self.seq_len = seq_len
        self.horizon = horizon
        self.feature_cols = feature_cols
        self.flip_y = flip_y
        self.scaler = scaler

        # store groups (player, match level)
        self.groups = []
        grouped = df.groupby(["match_id", "player_id"])

        # coords = g[list(coord_cols)].values.astype(np.float32)

        for (match, pid), g in grouped:
            g = g.reset_index(drop=True)
            if len(g) < seq_len + horizon:
                continue

            #co_ordinates
            coords = g[list(coord_cols)].values.astype(np.float32)

            if flip_y:
                coords[:, 1] = 1.0 - coords[:, 1]

            # features
            feats = g[feature_cols].values.astype(np.float32)

            if self.scaler is not None:
                feats = self.scaler.transform(feats)

            # precompute zones
            zones = xy_to_zone_vectorized(
                g["x_normalized"].values,
                g["y_normalized"].values,
                N_ROWS, N_COLS,
                flip_y=False
            )

            self.groups.append({
                "match": match,
                "player": pid,
                "features": feats,
                "zones": zones,
                "coords": coords
            })

        # index mapping (dataset idx → group, start_idx)
        self.index_map = []
        for gi, group in enumerate(self.groups):
            n = len(group["features"])
            valid_starts = n - (seq_len + horizon) + 1
            for start in range(valid_starts):
                self.index_map.append((gi, start))

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        gi, start = self.index_map[idx]
        group = self.groups[gi]

        # Input sequence
        seq = group["features"][start:start+self.seq_len]

        # Indices
        t_curr = start + self.seq_len - 1
        t_fut = t_curr + self.horizon

        targets= {}

        X_COL = self.feature_cols.index("x_normalized")
        Y_COL = self.feature_cols.index("y_normalized")

        if (CO_ORDINATES):
            # curr_xy = group["coords"][t_curr]
            # fut_xy = group["coords"][t_fut]
            # delta = fut_xy - curr_xy
            # targets["delta"] = torch.tensor(delta, dtype=torch.float32)
            fut_xy = group["coords"][t_fut]
            targets["coords"] = torch.tensor(fut_xy, dtype=torch.float32)
        else:
            zone = group["zones"][t_fut]
            targets["zone"] = torch.tensor(zone, dtype=torch.long)



        return torch.tensor(seq, dtype=torch.float32), targets

## Keras Sequence & Keras

In [ ]:
def keras_sequence_generator(df, feature_cols, scaler, seq_len, horizon,
                             coordinate_targets, n_rows=None, n_cols=None):
    """
    Generator that yields sequences one by one (or in small batches) instead of storing all.
    """
    for _, player_df in df.groupby(["match_id", "player_id"]):
        feats = player_df[feature_cols].values.astype(np.float32)
        if scaler is not None:
            feats = scaler.transform(feats)
        coords = player_df[list(coord_cols)].values.astype(np.float32)

        if coordinate_targets:
            # regression: dx, dy
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_curr = i + seq_len - 1
                t_fut = t_curr + horizon
                # delta = coords[t_fut] - coords[t_curr]
                # yield seq, delta.astype(np.float32)
                yield seq, coords[t_fut].astype(np.float32)

        else:
            zones = xy_to_zone_vectorized(
                player_df["x_normalized"].values,
                player_df["y_normalized"].values,
                n_rows, n_cols
            )
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_fut = i + seq_len - 1 + horizon
                zone = zones[t_fut]
                # yield seq, np.array([zone], dtype=np.int32)
                yield seq, np.int32(zone)
print("Keras_seq_gen_done")

Keras_seq_gen_done


In [ ]:
def make_tf_dataset(df, batch_size, shuffle=False, coordinate_targets=False):

    min_frames_needed = SEQ_LEN + HORIZON_FRAMES
    df = df.groupby(["match_id", "player_id"]).filter(
        lambda x: len(x) >= min_frames_needed
    )
    """Optimized tf.data pipeline with parallel processing and known cardinality"""
    output_types = (tf.float32, tf.float32 if coordinate_targets else tf.int32)
    output_shapes = (
    (SEQ_LEN, len(FEATURE_COLS)),  # input sequence shape
    (2,) if coordinate_targets else ()  # regression: (2,), classification: scalar ()
)


    # 🔧 PRE-CALCULATE: Count sequences for progress bar
    total_sequences = 0
    for _, player_df in df.groupby(["match_id", "player_id"]):
        total_sequences += max(0, len(player_df) - SEQ_LEN - HORIZON_FRAMES + 1)

    expected_batches = total_sequences // batch_size
    print(f"   Dataset: {total_sequences:,} sequences → {expected_batches} batches")

    ds = tf.data.Dataset.from_generator(
        lambda: keras_sequence_generator(
            df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,
            coordinate_targets, n_rows=N_ROWS, n_cols=N_COLS
        ),
        output_types=output_types,
        output_shapes=output_shapes
    )

    if shuffle:
        ds = ds.shuffle(4096)  # 🔧 INCREASED: Better randomization

    # 🔧 OPTIMIZATION: Batch before cache for memory efficiency
    # ensure labels are always correct shape

    def map_fn(x, y):
        if coordinate_targets:
            y = tf.reshape(y, [-1])
        return x, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds
print("make_tf_done")

make_tf_done


## Velocity addition Func

In [ ]:

def add_velocity_features(df):
    """Enhanced velocity and movement features for better coordinate prediction"""
    df = df.sort_values(["match_id", "player_id", "frame_number"])

    # === BASIC VELOCITY (existing) ===
    df["dx"] = df.groupby(["match_id", "player_id"])["x_normalized"].diff().fillna(0)
    df["dy"] = df.groupby(["match_id", "player_id"])["y_normalized"].diff().fillna(0)

    # Smooth velocities
    df["dx"] = df.groupby(["match_id", "player_id"])["dx"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)
    df["dy"] = df.groupby(["match_id", "player_id"])["dy"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)

    # === MULTI-WINDOW VELOCITY (NEW) ===
    for window in [3, 5, 10]:
        df[f"dx_avg_{window}"] = df.groupby(["match_id", "player_id"])["dx"].rolling(window, min_periods=1).mean().reset_index(level=[0,1], drop=True)
        df[f"dy_avg_{window}"] = df.groupby(["match_id", "player_id"])["dy"].rolling(window, min_periods=1).mean().reset_index(level=[0,1], drop=True)
        df[f"speed_avg_{window}"] = np.sqrt(df[f"dx_avg_{window}"]**2 + df[f"dy_avg_{window}"]**2)

    # === EXISTING FEATURES ===
    df["speed_normalized"] = np.sqrt(df["dx"]**2 + df["dy"]**2)
    df["acceleration"] = df.groupby(["match_id", "player_id"])["speed_normalized"].diff().fillna(0)
    df["movement_angle"] = np.arctan2(df["dy"], df["dx"])

    # === NEW: ACCELERATION TRENDS ===
    df["acceleration_trend"] = df.groupby(["match_id", "player_id"])["acceleration"].rolling(5, min_periods=1).mean().reset_index(level=[0,1], drop=True)

    # === NEW: DIRECTION PERSISTENCE ===
    df["angle_change"] = df.groupby(["match_id", "player_id"])["movement_angle"].diff().fillna(0)
    df["angle_stability"] = df.groupby(["match_id", "player_id"])["angle_change"].rolling(5, min_periods=1).std().reset_index(level=[0,1], drop=True).fillna(0)

    # === NEW: SPEED MOMENTUM ===
    df["speed_change_rate"] = df.groupby(["match_id", "player_id"])["speed_normalized"].diff().fillna(0)

    # === SPATIAL FEATURES (existing) ===
    df["distance_from_center"] = np.sqrt((df["x_normalized"] - 0.5)**2 + (df["y_normalized"] - 0.5)**2)
    df["distance_from_goal_home"] = df["x_normalized"]
    df["distance_from_goal_away"] = 1 - df["x_normalized"]
    df["distance_from_sideline"] = np.minimum(df["y_normalized"], 1 - df["y_normalized"])

    # === BALL & TEAM FEATURES (existing) ===
    if "team_spread" not in df.columns:
        df["team_spread"] = df.groupby(["match_id", "frame_number", "team_type"])["x_normalized"].transform("std").fillna(0)
    if "distance_to_ball" not in df.columns:
        df["distance_to_ball"] = 0.5
    if "ball_possession_proximity" not in df.columns:
        df["ball_possession_proximity"] = df["distance_to_ball"]

    print("\n📌 Enhanced velocity features added!")
    print(f"New multi-window features: dx_avg_3/5/10, dy_avg_3/5/10, speed_avg_3/5/10")
    print(f"New trend features: acceleration_trend, angle_change, angle_stability, speed_change_rate")

    return df

In [ ]:


# def add_velocity_features(df):
#     """Add velocity and enhanced features for better zone prediction"""
#     df = df.sort_values(["match_id", "player_id", "frame_number"])

#     # Basic velocity features
#     df["dx"] = df.groupby(["match_id", "player_id"])["x_normalized"].diff().fillna(0)
#     df["dy"] = df.groupby(["match_id", "player_id"])["y_normalized"].diff().fillna(0)

#     # Smooth velocities to reduce noise
#     df["dx"] = df.groupby(["match_id", "player_id"])["dx"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)
#     df["dy"] = df.groupby(["match_id", "player_id"])["dy"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)

#     # Speed and acceleration
#     df["speed_normalized"] = np.sqrt(df["dx"]**2 + df["dy"]**2)
#     df["acceleration"] = df.groupby(["match_id", "player_id"])["speed_normalized"].diff().fillna(0)

#     # Direction of movement (angle in radians)
#     df["movement_angle"] = np.arctan2(df["dy"], df["dx"])

#     # Distance from center of field
#     df["distance_from_center"] = np.sqrt((df["x_normalized"] - 0.5)**2 + (df["y_normalized"] - 0.5)**2)

#     # Proximity to field boundaries
#     df["distance_from_goal_home"] = df["x_normalized"]  # distance from home goal (x=0)
#     df["distance_from_goal_away"] = 1 - df["x_normalized"]  # distance from away goal (x=1)
#     df["distance_from_sideline"] = np.minimum(df["y_normalized"], 1 - df["y_normalized"])

#     # Team-based features (if available)
#     if "team_has_possession" in df.columns:
#         # Time since possession change
#         df["possession_change"] = df.groupby(["match_id"])["team_has_possession"].diff().fillna(0)
#         df["frames_since_possession_change"] = df.groupby(["match_id"]).apply(
#             lambda x: (x["possession_change"] != 0).cumsum()
#         ).reset_index(level=0, drop=True)

#     # Player density in surrounding area (simplified)
#     if "team_spread" not in df.columns:
#         # Calculate a simple team spread measure
#         df["team_spread"] = df.groupby(["match_id", "frame_number", "team_type"])["x_normalized"].transform("std").fillna(0)

#     # Ball proximity features
#     if "distance_to_ball" not in df.columns:
#         df["distance_to_ball"] = 0.5  # placeholder if not available

#     if "ball_possession_proximity" not in df.columns:
#         df["ball_possession_proximity"] = df["distance_to_ball"]  # simplified version
#     print("\n📌 add_velocity_features sample:")
#     print(df.head(5)[["x_normalized", "y_normalized","dx",
#                       "dy","speed_normalized", "acceleration","movement_angle"]])
#     return df

def add_contextual_features(df):
    """Add contextual features based on game state"""
    # Time-based features
    df["period_progress"] = df["timestamp_seconds"] / df.groupby(["match_id", "period"])["timestamp_seconds"].transform("max")

    # Formation-based features (simplified)
    if "position_category" in df.columns:
        # Create position-based features
        position_encoding = {"goalkeeper": 0, "defender": 1, "midfielder": 2, "forward": 3, "unknown": 4}
        df["position_encoded"] = df["position_category"].map(position_encoding).fillna(4)
    else:
        df["position_encoded"] = 2  # default to midfielder
    print("\n📌 add_contextual_features sample:")
    print(df.head(5)[[
        "period_progress",
        "position_encoded"
    ]])

    return df

## Original Locus Lab TCN

In [ ]:


# 1. The Helper Block (Padding)
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

# 2. The Residual Block
class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

# 3. The Main TCN Backbone (Renamed to avoid conflict)
class LocusLabTCN(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(LocusLabTCN, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                     dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size,
                                     dropout=dropout)]

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# 4. Your Final Model Wrapper
class DeltaTCN(nn.Module):
    def __init__(self, input_dim, num_classes=2, channels=[64, 128, 256, 512], kernel_size=3, dropout=0.3):
        super().__init__()
        # Initialize the specific TCN class defined above
        self.tcn = LocusLabTCN(input_dim, channels, kernel_size=kernel_size, dropout=dropout)
        self.co_coords = CO_ORDINATES

        # Output heads
        if self.co_coords:
            self.delta_head = nn.Linear(channels[-1], num_classes)          # dx, dy
        else:
            self.zone_head = nn.Linear(channels[-1], NUM_ZONES)   # classification over 144 zones

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = x.transpose(1, 2)  # (batch, input_dim, seq_len)
        out = self.tcn(x)
        out_last = out[:, :, -1]  # take last timestep for prediction

        if self.co_coords:
            return self.delta_head(out_last)      # regression: dx, dy
        else:
            return self.zone_head(out_last)       # classification: zones

## Keras TCN enhanced

In [ ]:


# @misc{KerasTCN,
#   author = {Philippe Remy},
#   title = {Temporal Convolutional Networks for Keras},
#   year = {2020},
#   publisher = {GitHub},
#   journal = {GitHub repository},
#   howpublished = {\url{https://github.com/philipperemy/keras-tcn}},
# }

try:
    # pylint: disable=E0611,E0401
    from keras.src.saving import register_keras_serializable  # For recent Keras
except ImportError:
    # pylint: disable=E0611,E0401
    from tensorflow.keras.saving import register_keras_serializable  # For older versions

# pylint: disable=E0611,E0401
from tensorflow.keras import backend as K, Model, Input, optimizers
# pylint: disable=E0611,E0401
from tensorflow.keras import layers
# pylint: disable=E0611,E0401
from tensorflow.keras.layers import Activation, SpatialDropout1D, Lambda
# pylint: disable=E0611,E0401
from tensorflow.keras.layers import Layer, Conv1D, Dense, BatchNormalization, LayerNormalization


def is_power_of_two(num: int):
    return num != 0 and ((num & (num - 1)) == 0)


def adjust_dilations(dilations: list):
    if all([is_power_of_two(i) for i in dilations]):
        return dilations
    else:
        new_dilations = [2 ** i for i in dilations]
        return new_dilations


class ResidualBlock(Layer):

    def __init__(self,
                 dilation_rate: int,
                 nb_filters: int,
                 kernel_size: int,
                 padding: str,
                 activation: str = 'relu',
                 dropout_rate: float = 0,
                 kernel_initializer: str = 'he_normal',
                 use_batch_norm: bool = False,
                 use_layer_norm: bool = False,
                 **kwargs):
        """Defines the residual block for the WaveNet TCN
        Args:
            x: The previous layer in the model
            training: boolean indicating whether the layer should behave in training mode or in inference mode
            dilation_rate: The dilation power of 2 we are using for this residual block
            nb_filters: The number of convolutional filters to use in this block
            kernel_size: The size of the convolutional kernel
            padding: The padding used in the convolutional layers, 'same' or 'causal'.
            activation: The final activation used in o = Activation(x + F(x))
            dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
            kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
            use_batch_norm: Whether to use batch normalization in the residual layers or not.
            use_layer_norm: Whether to use layer normalization in the residual layers or not.
            kwargs: Any initializers for Layer class.
        """

        self.dilation_rate = dilation_rate
        self.nb_filters = nb_filters
        self.kernel_size = kernel_size
        self.padding = padding
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_batch_norm = use_batch_norm
        self.use_layer_norm = use_layer_norm
        self.kernel_initializer = kernel_initializer
        self.layers = []
        self.shape_match_conv = None
        self.res_output_shape = None
        self.final_activation = None
        self.batch_norm_layers = []
        self.layer_norm_layers = []

        super(ResidualBlock, self).__init__(**kwargs)

    def _build_layer(self, layer):
        """Helper function for building layer
        Args:
            layer: Appends layer to internal layer list and builds it based on the current output
                   shape of ResidualBlocK. Updates current output shape.
        """
        self.layers.append(layer)
        self.layers[-1].build(self.res_output_shape)
        self.res_output_shape = self.layers[-1].compute_output_shape(self.res_output_shape)

    def build(self, input_shape):

        with K.name_scope(self.name):  # name scope used to make sure weights get unique names
            self.layers = []
            self.res_output_shape = input_shape

            for k in range(2):  # dilated conv block.
                name = 'conv1D_{}'.format(k)
                with K.name_scope(name):  # name scope used to make sure weights get unique names
                    conv = Conv1D(
                        filters=self.nb_filters,
                        kernel_size=self.kernel_size,
                        dilation_rate=self.dilation_rate,
                        padding=self.padding,
                        name=name,
                        kernel_initializer=self.kernel_initializer
                    )
                    self._build_layer(conv)

                with K.name_scope('norm_{}'.format(k)):
                    if self.use_batch_norm:
                        bn_layer = BatchNormalization()
                        self.batch_norm_layers.append(bn_layer)
                        self._build_layer(bn_layer)
                    elif self.use_layer_norm:
                        ln_layer = LayerNormalization()
                        self.layer_norm_layers.append(ln_layer)
                        self._build_layer(ln_layer)

                with K.name_scope('act_and_dropout_{}'.format(k)):
                    self._build_layer(Activation(self.activation, name='Act_Conv1D_{}'.format(k)))
                    self._build_layer(SpatialDropout1D(rate=self.dropout_rate, name='SDropout_{}'.format(k)))

            if self.nb_filters != input_shape[-1]:
                # 1x1 conv to match the shapes (channel dimension).
                name = 'matching_conv1D'
                with K.name_scope(name):
                    # make and build this layer separately because it directly uses input_shape.
                    # 1x1 conv.
                    self.shape_match_conv = Conv1D(
                        filters=self.nb_filters,
                        kernel_size=1,
                        padding='same',
                        name=name,
                        kernel_initializer=self.kernel_initializer
                    )
            else:
                name = 'matching_identity'
                self.shape_match_conv = Lambda(lambda x: x, name=name)

            with K.name_scope(name):
                self.shape_match_conv.build(input_shape)
                self.res_output_shape = self.shape_match_conv.compute_output_shape(input_shape)

            self._build_layer(Activation(self.activation, name='Act_Conv_Blocks'))
            self.final_activation = Activation(self.activation, name='Act_Res_Block')
            self.final_activation.build(self.res_output_shape)  # probably isn't necessary

            # this is done to force Keras to add the layers in the list to self._layers
            for layer in self.layers:
                self.__setattr__(layer.name, layer)
            self.__setattr__(self.shape_match_conv.name, self.shape_match_conv)
            self.__setattr__(self.final_activation.name, self.final_activation)

            super(ResidualBlock, self).build(input_shape)  # done to make sure self.built is set True

    def call(self, inputs, training=None, **kwargs):
        """
        Returns: A tuple where the first element is the residual model tensor, and the second
                 is the skip connection tensor.
        """
        # https://arxiv.org/pdf/1803.01271.pdf  page 4, Figure 1 (b).
        # x1: Dilated Conv -> Norm -> Dropout (x2).
        # x2: Residual (1x1 matching conv - optional).
        # Output: x1 + x2.
        # x1 -> connected to skip connections.
        # x1 + x2 -> connected to the next block.
        #       input
        #     x1      x2
        #   conv1D    1x1 Conv1D (optional)
        #    ...
        #   conv1D
        #    ...
        #       x1 + x2
        x1 = inputs
        for layer in self.layers:
            training_flag = 'training' in dict(inspect.signature(layer.call).parameters)
            x1 = layer(x1, training=training) if training_flag else layer(x1)
        x2 = self.shape_match_conv(inputs)
        x1_x2 = self.final_activation(layers.add([x2, x1], name='Add_Res'))
        return [x1_x2, x1]

    def compute_output_shape(self, input_shape):
        return [self.res_output_shape, self.res_output_shape]


@register_keras_serializable()
class TCN(Layer):
    """Creates a TCN layer.

        Input shape:
            A 3D tensor with shape (batch_size, timesteps, input_dim).

        Args:
            nb_filters: The number of filters to use in the convolutional layers. Can be a list.
            kernel_size: The size of the kernel to use in each convolutional layer.
            dilations: The list of the dilations. Example is: [1, 2, 4, 8, 16, 32, 64].
            nb_stacks : The number of stacks of residual blocks to use.
            padding: The padding to use in the convolutional layers, 'causal' or 'same'.
            use_skip_connections: Boolean. If we want to add skip connections from input to each residual blocK.
            return_sequences: Boolean. Whether to return the last output in the output sequence, or the full sequence.
            activation: The activation used in the residual blocks o = Activation(x + F(x)).
            dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
            kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
            use_batch_norm: Whether to use batch normalization in the residual layers or not.
            use_layer_norm: Whether to use layer normalization in the residual layers or not.
            go_backwards: Boolean (default False). If True, process the input sequence backwards and
            return the reversed sequence.
            return_state: Boolean. Whether to return the last state in addition to the output. Default: False.
            kwargs: Any other arguments for configuring parent class Layer. For example "name=str", Name of the model.
                    Use unique names when using multiple TCN.
        Returns:
            A TCN layer.
        """

    def __init__(self,
                 nb_filters=64,
                 kernel_size=3,
                 nb_stacks=1,
                 dilations=(1, 2, 4, 8, 16, 32),
                 padding='causal',
                 use_skip_connections=True,
                 dropout_rate=0.0,
                 return_sequences=False,
                 activation='relu',
                 kernel_initializer='he_normal',
                 use_batch_norm=False,
                 use_layer_norm=False,
                 go_backwards=False,
                 return_state=False,
                 **kwargs):
        self.stateful = False  # TCN are not stateful. Keras requires this parameter.
        self.return_sequences = return_sequences
        self.dropout_rate = dropout_rate
        self.use_skip_connections = use_skip_connections
        self.dilations = dilations
        self.nb_stacks = nb_stacks
        self.kernel_size = kernel_size
        self.nb_filters = nb_filters
        self.activation_name = activation
        self.padding = padding
        self.kernel_initializer = kernel_initializer
        self.use_batch_norm = use_batch_norm
        self.use_layer_norm = use_layer_norm
        self.go_backwards = go_backwards
        self.return_state = return_state
        self.skip_connections = []
        self.residual_blocks = []
        self.layers_outputs = []
        self.build_output_shape = None
        self.slicer_layer = None  # in case return_sequence=False
        self.output_slice_index = None  # in case return_sequence=False
        self.padding_same_and_time_dim_unknown = False  # edge case if padding='same' and time_dim = None

        if self.use_batch_norm + self.use_layer_norm > 1:
            raise ValueError('Only one normalization can be specified at once.')

        if isinstance(self.nb_filters, list):
            assert len(self.nb_filters) == len(self.dilations)
            if len(set(self.nb_filters)) > 1 and self.use_skip_connections:
                raise ValueError('Skip connections are not compatible '
                                 'with a list of filters, unless they are all equal.')

        if padding != 'causal' and padding != 'same':
            raise ValueError("Only 'causal' or 'same' padding are compatible for this layer.")

        # initialize parent class
        super(TCN, self).__init__(**kwargs)

    @property
    def receptive_field(self):
        return 1 + 2 * (self.kernel_size - 1) * self.nb_stacks * sum(self.dilations)

    def tolist(self, shape):
        # noinspection PyBroadException
        try:
            return shape.as_list()
        except Exception:
            return list(shape)

    def build(self, input_shape):

        # member to hold current output shape of the layer for building purposes
        self.build_output_shape = input_shape

        # list to hold all the member ResidualBlocks
        self.residual_blocks = []
        total_num_blocks = self.nb_stacks * len(self.dilations)
        if not self.use_skip_connections:
            total_num_blocks += 1  # cheap way to do a false case for below

        for s in range(self.nb_stacks):
            for i, d in enumerate(self.dilations):
                res_block_filters = self.nb_filters[i] if isinstance(self.nb_filters, list) else self.nb_filters
                self.residual_blocks.append(ResidualBlock(dilation_rate=d,
                                                          nb_filters=res_block_filters,
                                                          kernel_size=self.kernel_size,
                                                          padding=self.padding,
                                                          activation=self.activation_name,
                                                          dropout_rate=self.dropout_rate,
                                                          use_batch_norm=self.use_batch_norm,
                                                          use_layer_norm=self.use_layer_norm,
                                                          kernel_initializer=self.kernel_initializer,
                                                          name='residual_block_{}'.format(len(self.residual_blocks))))
                # build newest residual block
                self.residual_blocks[-1].build(self.build_output_shape)
                self.build_output_shape = self.residual_blocks[-1].res_output_shape

        # this is done to force keras to add the layers in the list to self._layers
        for layer in self.residual_blocks:
            self.__setattr__(layer.name, layer)

        self.output_slice_index = None
        if self.padding == 'same':
            time = self.tolist(self.build_output_shape)[1]
            if time is not None:  # if time dimension is defined. e.g. shape = (bs, 500, input_dim).
                self.output_slice_index = int(self.tolist(self.build_output_shape)[1] / 2)
            else:
                # It will known at call time. c.f. self.call.
                self.padding_same_and_time_dim_unknown = True

        else:
            self.output_slice_index = -1  # causal case.
        self.slicer_layer = Lambda(lambda tt: tt[:, self.output_slice_index, :], name='Slice_Output')
        self.slicer_layer.build(self.tolist(self.build_output_shape))

    def compute_output_shape(self, input_shape):
        """
        Overridden in case keras uses it somewhere... no idea. Just trying to avoid future errors.
        """
        if not self.built:
            self.build(input_shape)
        if not self.return_sequences:
            batch_size = self.build_output_shape[0]
            batch_size = batch_size.value if hasattr(batch_size, 'value') else batch_size
            nb_filters = self.build_output_shape[-1]
            return [batch_size, nb_filters]
        else:
            # Compatibility tensorflow 1.x
            return [v.value if hasattr(v, 'value') else v for v in self.build_output_shape]

    def call(self, inputs, training=None, **kwargs):
        x = inputs

        if self.go_backwards:
            # reverse x in the time axis
            x = tf.reverse(x, axis=[1])

        self.layers_outputs = [x]
        self.skip_connections = []
        for res_block in self.residual_blocks:
            try:
                x, skip_out = res_block(x, training=training)
            except TypeError:  # compatibility with tensorflow 1.x
                x, skip_out = res_block(K.cast(x, 'float32'), training=training)
            self.skip_connections.append(skip_out)
            self.layers_outputs.append(x)

        if self.use_skip_connections:
            if len(self.skip_connections) > 1:
                # Keras: A merge layer should be called on a list of at least 2 inputs. Got 1 input.
                x = layers.add(self.skip_connections, name='Add_Skip_Connections')
            else:
                x = self.skip_connections[0]
            self.layers_outputs.append(x)

        if not self.return_sequences:
            # case: time dimension is unknown. e.g. (bs, None, input_dim).
            if self.padding_same_and_time_dim_unknown:
                self.output_slice_index = K.shape(self.layers_outputs[-1])[1] // 2
            x = self.slicer_layer(x)
            self.layers_outputs.append(x)
        return x

    def get_config(self):
        """
        Returns the config of a the layer. This is used for saving and loading from a model
        :return: python dictionary with specs to rebuild layer
        """
        config = super(TCN, self).get_config()
        config['nb_filters'] = self.nb_filters
        config['kernel_size'] = self.kernel_size
        config['nb_stacks'] = self.nb_stacks
        config['dilations'] = self.dilations
        config['padding'] = self.padding
        config['use_skip_connections'] = self.use_skip_connections
        config['dropout_rate'] = self.dropout_rate
        config['return_sequences'] = self.return_sequences
        config['activation'] = self.activation_name
        config['use_batch_norm'] = self.use_batch_norm
        config['use_layer_norm'] = self.use_layer_norm
        config['kernel_initializer'] = self.kernel_initializer
        config['go_backwards'] = self.go_backwards
        config['return_state'] = self.return_state
        return config


def compiled_tcn(num_feat,  # type: int
                 num_classes,  # type: int
                 nb_filters,  # type: int
                 kernel_size,  # type: int
                 dilations,  # type: List[int]
                 nb_stacks,  # type: int
                 max_len,  # type: int
                 output_len=1,  # type: int
                 padding='causal',  # type: str
                 use_skip_connections=False,  # type: bool
                 return_sequences=True,
                 regression=False,  # type: bool
                 dropout_rate=0.05,  # type: float
                 name='tcn',  # type: str,
                 kernel_initializer='he_normal',  # type: str,
                 activation='relu',  # type:str,
                 opt='adam',
                 lr=0.002,
                 use_batch_norm=False,
                 use_layer_norm=False):
    # type: (...) -> Model
    """Creates a compiled TCN model for a given task (i.e. regression or classification).
    Classification uses a sparse categorical loss. Please input class ids and not one-hot encodings.

    Args:
        num_feat: The number of features of your input, i.e. the last dimension of: (batch_size, timesteps, input_dim).
        num_classes: The size of the final dense layer, how many classes we are predicting.
        nb_filters: The number of filters to use in the convolutional layers.
        kernel_size: The size of the kernel to use in each convolutional layer.
        dilations: The list of the dilations. Example is: [1, 2, 4, 8, 16, 32, 64].
        nb_stacks : The number of stacks of residual blocks to use.
        max_len: The maximum sequence length, use None if the sequence length is dynamic.
        padding: The padding to use in the convolutional layers.
        use_skip_connections: Boolean. If we want to add skip connections from input to each residual blocK.
        return_sequences: Boolean. Whether to return the last output in the output sequence, or the full sequence.
        regression: Whether the output should be continuous or discrete.
        dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
        activation: The activation used in the residual blocks o = Activation(x + F(x)).
        name: Name of the model. Useful when having multiple TCN.
        kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
        opt: Optimizer name.
        lr: Learning rate.
        use_batch_norm: Whether to use batch normalization in the residual layers or not.
        use_layer_norm: Whether to use layer normalization in the residual layers or not.
    Returns:
        A compiled keras TCN.
    """

    dilations = adjust_dilations(dilations)

    input_layer = Input(shape=(max_len, num_feat))

    x = TCN(nb_filters, kernel_size, nb_stacks, dilations, padding,
            use_skip_connections, dropout_rate, return_sequences,
            activation, kernel_initializer, use_batch_norm, use_layer_norm,
            name=name)(input_layer)

    print('x.shape=', x.shape)

    def get_opt():
        if opt == 'adam':
            return optimizers.Adam(learning_rate=lr, clipnorm=1.)
        elif opt == 'rmsprop':
            return optimizers.RMSprop(learning_rate=lr, clipnorm=1.)
        else:
            raise Exception('Only Adam and RMSProp are available here')

    if not regression:
        # classification
        x = Dense(num_classes)(x)
        x = Activation('softmax')(x)
        output_layer = x
        model = Model(input_layer, output_layer)

        # https://github.com/keras-team/keras/pull/11373
        # It's now in Keras@master but still not available with pip.
        # TODO remove later.
        # def accuracy(y_true, y_pred):
        #    if K.ndim(y_true) > 1:
        #         y_true = K.squeeze(y_true, axis=-1)

        #     # Ensure integer labels
        #     y_true = K.cast(y_true, 'int32')

        #     # Predicted class indices
        #     y_pred_labels = K.argmax(y_pred, axis=-1)
        #     y_pred_labels = K.cast(y_pred_labels, 'int32')

        # return K.mean(K.cast(K.equal(y_true, y_pred_labels), K.floatx()))


        model.compile(get_opt(), loss='sparse_categorical_crossentropy', metrics=["sparse_categorical_accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)  ])
    else:
        # regression
        x = Dense(num_classes)(x)
        x = Activation('sigmoid')(x)

        output_layer = x
        model = Model(input_layer, output_layer)
        model.compile(get_opt(), loss='mean_squared_error', metrics=['mae'])
    print('model.x = {}'.format(input_layer.shape))
    print('model.y = {}'.format(output_layer.shape))
    return model


def tcn_full_summary(model: Model, expand_residual_blocks=True):
    import tensorflow as tf
    # 2.6.0-rc1, 2.5.0...
    versions = [int(v) for v in tf.__version__.split('-')[0].split('.')]
    if versions[0] <= 2 and versions[1] < 5:
        layers = model._layers.copy()  # store existing layers
        model._layers.clear()  # clear layers

        for i in range(len(layers)):
            if isinstance(layers[i], TCN):
                for layer in layers[i]._layers:
                    if not isinstance(layer, ResidualBlock):
                        if not hasattr(layer, '__iter__'):
                            model._layers.append(layer)
                    else:
                        if expand_residual_blocks:
                            for lyr in layer._layers:
                                if not hasattr(lyr, '__iter__'):
                                    model._layers.append(lyr)
                        else:
                            model._layers.append(layer)
            else:
                model._layers.append(layers[i])

        model.summary()  # print summary

        # restore original layers
        model._layers.clear()
        [model._layers.append(lyr) for lyr in layers]
    else:
        print('WARNING: tcn_full_summary: Compatible with tensorflow 2.5.0 or below.')
        print('Use tensorboard instead. Example in keras-tcn/tasks/tcn_tensorboard.py.')

In [ ]:
#CUSTOM LOSS FUNCTIONS FOR BETTER COORDINATE PREDICTION

def huber_loss_tf(delta=0.1):
    """
    Huber loss - less sensitive to outliers than MSE
    Better for coordinate prediction with occasional large errors
    """
    def loss(y_true, y_pred):
        error = y_true - y_pred
        is_small = tf.abs(error) <= delta
        squared_loss = 0.5 * tf.square(error)
        linear_loss = delta * (tf.abs(error) - 0.5 * delta)
        return tf.where(is_small, squared_loss, linear_loss)
    return loss

def weighted_mse_loss():
    """
    Weighted MSE that penalizes x and y errors differently
    (x-axis errors might be more critical in football)
    """
    def loss(y_true, y_pred):
        x_error = tf.square(y_true[:, 0] - y_pred[:, 0])
        y_error = tf.square(y_true[:, 1] - y_pred[:, 1])
        return 1.2 * x_error + 0.8 * y_error  # weight x-direction more
    return loss

print("✓ Custom loss functions loaded")

✓ Custom loss functions loaded


##     Enhanced model combining LSTM for temporal patterns and CNN for feature extraction


In [ ]:
class EnhancedFootballModel(nn.Module):
    """
    Enhanced model combining LSTM for temporal patterns and CNN for feature extraction
    """
    def __init__(self, input_dim, num_classes, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()

        # Feature extraction layer
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # LSTM for temporal modeling
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # Attention mechanism to focus on important time steps
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,  # bidirectional
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )

        # Final classification layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LSTM):
                for name, param in module.named_parameters():
                    if 'weight' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)

    # def forward(self, x):
    #     # x shape: (batch_size, seq_len, input_dim)
    #     batch_size, seq_len, input_dim = x.shape

    #     # Extract features for each time step
    #     x_reshaped = x.view(-1, input_dim)  # (batch_size * seq_len, input_dim)
    #     features = self.feature_extractor(x_reshaped)  # (batch_size * seq_len, hidden_dim)
    #     features = features.view(batch_size, seq_len, -1)  # (batch_size, seq_len, hidden_dim)

    #     # LSTM processing
    #     lstm_out, (hidden, cell) = self.lstm(features)  # (batch_size, seq_len, hidden_dim * 2)

    #     # Attention mechanism
    #     attended_out, attention_weights = self.attention(lstm_out, lstm_out, lstm_out)

    #     # Use the last time step for classification
    #     final_features = attended_out[:, -1, :]  # (batch_size, hidden_dim * 2)

    #     # Classification
    #     logits = self.classifier(final_features)  # (batch_size, num_classes)

    #     return logits

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len, input_dim = x.shape

        # 1. Feature extraction for each time step
        x_reshaped = x.view(-1, input_dim)
        features = self.feature_extractor(x_reshaped)
        features = features.view(batch_size, seq_len, -1)

        # 2. LSTM processing
        lstm_out, _ = self.lstm(features) # (batch_size, seq_len, hidden_dim * 2)

        # 3. Attention mechanism
        # attended_out shape: (batch_size, seq_len, hidden_dim * 2)
        attended_out, _ = self.attention(lstm_out, lstm_out, lstm_out)

        # 4. Global Average Pooling (The upgrade)
        # We take the mean across the sequence (dim 1)
        # This captures the "essence" of the whole sequence after attention weights it
        avg_pool = torch.mean(attended_out, dim=1)
        max_pool, _ = torch.max(attended_out, dim=1)

        # Combining Avg and Max pooling is a common trick to get the best of both worlds
        final_features = avg_pool + max_pool

        # 5. Classification
        logits = self.classifier(final_features)

        return logits


## Improved Temporal Convolutional Network (not verified)

In [ ]:
 class TemporalConvNet(nn.Module):
    """Improved Temporal Convolutional Network"""
    def __init__(self, input_dim, num_classes, num_channels=[64, 128, 256], dropout=0.3):
        super().__init__()
        layers = []
        in_ch = input_dim

        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i  # increasing dilation
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=dilation, dilation=dilation),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_ch = out_ch

        self.network = nn.Sequential(*layers)

        # Global pooling and classification
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(in_ch, in_ch // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_ch // 2, num_classes)
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        x = x.transpose(1, 2)  # (batch_size, input_dim, seq_len)
        x = self.network(x)
        x = self.global_pool(x).squeeze(-1)  # (batch_size, channels)
        return self.classifier(x)


##  Unified training function for PyTorch or Keras backend.

In [ ]:
# def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE):

#     if BACKEND == "pytorch":

#         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         if torch.cuda.device_count() > 1:
#             model = torch.nn.DataParallel(model)
#         model.to(device)

#         optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
#         criterion = torch.nn.MSELoss() if CO_ORDINATES else torch.nn.CrossEntropyLoss()
#          # LR scheduler for classification
#         scheduler = None
#         if not CO_ORDINATES:
#             scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#                 optimizer, mode='max', factor=0.5, patience=patience // 2
#             )
#         best_val_metric = float('inf') if CO_ORDINATES else 0.0
#         patience_counter = 0
#         history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_top3_acc': []}

#         print(f"Starting training on {device} for {epochs} epochs...")

#         def get_targets(yb):
#             if isinstance(yb, dict):
#                 yb = {k: v.to(device) for k, v in yb.items()}
#                 return yb["delta"] if CO_ORDINATES else yb["zone"]
#             else:
#                 return yb.to(device)

#         for epoch in range(epochs):
#             model.train()
#             running_loss = 0.0
#             correct, total = 0, 0

#             for Xb, yb in train_loader:
#                 Xb = Xb.to(device)
#                 targets = get_targets(yb)

#                 optimizer.zero_grad()
#                 outputs = model(Xb)
#                 loss = criterion(outputs, targets)
#                 loss.backward()

#                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#                 optimizer.step()

#                 running_loss += loss.item() * Xb.size(0)

#                 if not CO_ORDINATES:
#                     _, preds = outputs.max(1)
#                     correct += (preds == targets).sum().item()
#                     total += targets.size(0)

#                 if (batch_idx + 1) % 100 == 0:
#                     if not CO_ORDINATES:
#                         batch_acc = correct / max(1, total)
#                         print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx+1}/{len(train_loader)}] "
#                           f"Loss: {loss.item():.4f} Acc: {batch_acc:.4f}")
#                     else:
#                         print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx+1}/{len(train_loader)}] Loss: {loss.item():.4f}")

#             train_loss = running_loss / len(train_loader.dataset)
#             train_acc = (correct / total) if not CO_ORDINATES else None

#             # Validation
#             model.eval()
#             val_loss = 0.0
#             val_correct, val_total = 0, 0
#             all_probs = []

#             with torch.no_grad():
#                 for Xb, yb in val_loader:
#                     Xb = Xb.to(device)
#                     targets = get_targets(yb)

#                     outputs = model(Xb)
#                     loss = criterion(outputs, targets)
#                     val_loss += loss.item() * Xb.size(0)

#                     if not CO_ORDINATES:
#                         probs = F.softmax(outputs, dim=1)
#                         all_probs.append(probs.cpu())
#                         _, preds = outputs.max(1)
#                         val_correct += (preds == targets).sum().item()
#                         val_total += targets.size(0)

#             val_loss /= len(val_loader.dataset)
#             val_acc = (val_correct / val_total) if not CO_ORDINATES else None


#             val_top3 = None
#             if not CO_ORDINATES and all_probs:
#                 val_top3 = top_k_accuracy_score(
#                     torch.cat(all_labels).numpy(),
#                     torch.cat(all_probs).numpy(),
#                     k=3
#                 )

#             history['train_loss'].append(train_loss)
#             history['train_acc'].append(train_acc)
#             history['val_acc'].append(val_acc)
#             history['val_top3_acc'].append(val_top3)

#             # Early stopping
#             current_metric = val_loss if CO_ORDINATES else val_acc
#             improved = (current_metric < best_val_metric) if CO_ORDINATES else (current_metric > best_val_metric)
#             if improved:
#                 best_val_metric = current_metric
#                 best_model_wts = copy.deepcopy(model.state_dict())
#                 patience_counter = 0
#                 print(f"New Best: {best_val_metric:.4f}")
#             else:
#                 patience_counter += 1

#             if not CO_ORDINATES and scheduler:
#                 scheduler.step(val_acc)

#             if patience_counter >= patience:
#                 break

#         model.load_state_dict(best_model_wts)
#         return model, history

#     elif BACKEND == "keras":
#         callbacks = [
#                 tf.keras.callbacks.EarlyStopping(
#                     monitor='val_mae' if CO_ORDINATES else 'val_accuracy',
#                     patience=5,
#                     restore_best_weights=True,
#                     mode='min' if CO_ORDINATES else 'max'
#                 ),
#                 tf.keras.callbacks.ReduceLROnPlateau(
#                     monitor='val_mae' if CO_ORDINATES else 'val_accuracy',
#                     factor=0.5,
#                     patience=3,
#                     min_lr=1e-6,
#                     mode='min' if CO_ORDINATES else 'max'
#                 ),
#                 tf.keras.callbacks.ModelCheckpoint(
#                     'best_model_mae.keras',
#                     monitor='val_mae' if CO_ORDINATES else 'val_accuracy',
#                     save_best_only=True,
#                     mode='min' if CO_ORDINATES else 'max'
#                 )
#             ]
#         history = model.fit(
#                 train_loader,
#                 validation_data=val_loader,
#                 epochs=epochs,
#                 callbacks=callbacks,
#                 verbose=1
#             )

#         return model, history

In [ ]:
class GarbageCollectorCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()
        tf.keras.backend.clear_session() # Resets the global state (useful for heavy models)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE):

    if BACKEND == "pytorch":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if torch.cuda.device_count() > 1:
            model = torch.nn.DataParallel(model)
        model.to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = torch.nn.MSELoss() if CO_ORDINATES else torch.nn.CrossEntropyLoss()

        scheduler = None
        if not CO_ORDINATES:
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=patience // 2
            )

        best_val_metric = float('inf') if CO_ORDINATES else 0.0
        patience_counter = 0
        history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_top5_acc': []}

        print(f"Starting training on {device} for {epochs} epochs...")

        def get_targets(yb):
            if isinstance(yb, dict):
                yb = {k: v.to(device) for k, v in yb.items()}
                return yb["coords"] if CO_ORDINATES else yb["zone"]
            else:
                return yb.to(device)

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            correct, total = 0, 0

            for batch_idx, (Xb, yb) in enumerate(train_loader):
                Xb = Xb.to(device)
                targets = get_targets(yb)

                optimizer.zero_grad()
                outputs = model(Xb)
                loss = criterion(outputs, targets)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item() * Xb.size(0)

                if not CO_ORDINATES:
                    _, preds = outputs.max(1)
                    correct += (preds == targets).sum().item()
                    total += targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    if not CO_ORDINATES:
                        batch_acc = correct / max(1, total)
                        print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx+1}/{len(train_loader)}] "
                              f"Loss: {loss.item():.4f} Acc: {batch_acc:.4f}")
                    else:
                        print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx+1}/{len(train_loader)}] Loss: {loss.item():.4f}")

            train_loss = running_loss / len(train_loader.dataset)
            train_acc = (correct / total) if not CO_ORDINATES else None

            # Validation
            model.eval()
            val_loss = 0.0
            val_correct, val_total = 0, 0
            all_probs = []

            with torch.no_grad():
                for Xb, yb in val_loader:
                    Xb = Xb.to(device)
                    targets = get_targets(yb)

                    outputs = model(Xb)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item() * Xb.size(0)

                    if not CO_ORDINATES:
                        probs = F.softmax(outputs, dim=1)
                        all_probs.append(probs.cpu())
                        _, preds = outputs.max(1)
                        val_correct += (preds == targets).sum().item()
                        val_total += targets.size(0)

            val_loss /= len(val_loader.dataset)
            val_acc = (val_correct / val_total) if not CO_ORDINATES else None

            val_top5 = None
            if not CO_ORDINATES and all_probs:
                val_top5 = top_k_accuracy_score(
                    torch.cat(all_labels).numpy(),
                    torch.cat(all_probs).numpy(),
                    k=3
                )

            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)
            history['val_top5_acc'].append(val_top5)

            # Explicitly delete large local tensors to break references
            del Xb, yb, outputs, loss, Xb_v, yb_v, outputs_v
            if not CO_ORDINATES:
                del all_probs, all_labels

            gc.collect() # Force Python to clear unused objects
            if torch.cuda.is_available():
                torch.cuda.empty_cache() # Release GPU memory back to the system

            # Early stopping
            current_metric = val_loss if CO_ORDINATES else val_acc
            improved = (current_metric < best_val_metric) if CO_ORDINATES else (current_metric > best_val_metric)
            if improved:
                best_val_metric = current_metric
                best_model_wts = copy.deepcopy(model.state_dict())
                patience_counter = 0
                print(f"New Best: {best_val_metric:.4f}")
            else:
                patience_counter += 1

            if not CO_ORDINATES and scheduler:
                scheduler.step(val_acc)

            if patience_counter >= patience:
                break

        model.load_state_dict(best_model_wts)
        return model, history

    elif BACKEND == "keras":
        # ✅ Model already compiled in strategy.scope() at line 2556
        # Just define callbacks and train

        callbacks = [
            GarbageCollectorCallback(),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_mae' if CO_ORDINATES else 'val_sparse_top_k_categorical_accuracy',
                patience=5,
                restore_best_weights=True,
                mode='min' if CO_ORDINATES else 'max'
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_mae' if CO_ORDINATES else 'val_sparse_top_k_categorical_accuracy',
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                mode='min' if CO_ORDINATES else 'max'
            ),
            tf.keras.callbacks.ModelCheckpoint(
                'best_model_mae.keras',
                monitor='val_mae' if CO_ORDINATES else 'val_sparse_top_k_categorical_accuracy',
                save_best_only=True,
                mode='min' if CO_ORDINATES else 'max'
            )
        ]

        history = model.fit(
            train_loader,
            validation_data=val_loader,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )
        print("\nSanity check: evaluate on TRAIN samples")
        model.evaluate(train_ds.take(200))

        return model, history

## Enhanced evaluation with multiple metrics      

In [ ]:
def evaluate_with_metrics(model, loader):

    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            probs = F.softmax(logits, dim=1)

            _, preds = logits.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
            all_probs.append(probs.cpu())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = torch.cat(all_probs, dim=0)

    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    top3_acc = top_k_accuracy_score(all_labels, all_probs.numpy(), k=3)

    return acc, top3_acc, all_probs


def evaluate_detailed(model, loader, zone_names=None):
    """Detailed evaluation with classification report"""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            _, preds = logits.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Accuracy metrics
    acc = accuracy_score(all_labels, all_preds)

    print(f"Overall Accuracy: {acc:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=zone_names))

    return acc, all_preds, all_labels


## Unified evaluation function for both regression (dx/dy) and classification (zones).

        
    Returns:
        - For regression: dict with 'mse', 'mae', 'rmse'
        - For classification: dict with 'accuracy', 'top3_accuracy', 'all_probs'

In [ ]:
def evaluate_model(model, loader, co_ordinates=CO_ORDINATES, zone_names=None):

    model.eval()
    device = next(model.parameters()).device  # get model device automatically

    if co_ordinates:
        # Regression
        all_preds, all_targets = [], []
        criterion_mse = nn.MSELoss(reduction='sum')
        criterion_mae = nn.L1Loss(reduction='sum')

        with torch.no_grad():
            for Xb, yb in loader:
                Xb = Xb.to(device)
                yb = {k: v.to(device) for k, v in yb.items()}  # regression dict
                targets = yb["delta"]

                outputs = model(Xb)
                if epoch == 0 and batch_idx == 0:
                    print("Outputs shape:", outputs.shape)
                    print("Target min:", targets_tensor.min().item(),"Target max:", targets_tensor.max().item())
                all_preds.append(outputs.cpu())
                all_targets.append(targets.cpu())

        all_preds = torch.cat(all_preds, dim=0)
        all_targets = torch.cat(all_targets, dim=0)

        mse = criterion_mse(all_preds, all_targets).item() / len(loader.dataset)
        mae = criterion_mae(all_preds, all_targets).item() / len(loader.dataset)
        rmse = mse ** 0.5

        metrics = {'mse': mse, 'mae': mae, 'rmse': rmse}

    else:
        # Classification
        all_preds, all_labels, all_probs = [], [], []

        with torch.no_grad():
            for Xb, yb in loader:
                Xb, yb = Xb.to(device), yb.to(device)
                outputs = model(Xb)
                if epoch == 0 and batch_idx == 0:
                    print("Outputs shape:", outputs.shape)
                    print("Target min:", targets_tensor.min().item(),"Target max:", targets_tensor.max().item())

                probs = F.softmax(outputs, dim=1)
                _, preds = outputs.max(1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(yb.cpu().numpy())
                all_probs.append(probs.cpu())

        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        all_probs = torch.cat(all_probs, dim=0)

        acc = accuracy_score(all_labels, all_preds)
        top3_acc = top_k_accuracy_score(all_labels, all_probs.numpy(), k=3)

        metrics = {'accuracy': acc, 'top3_accuracy': top3_acc, 'all_probs': all_probs}

        # Optional: print classification report
        if zone_names is not None:
            print("\nClassification Report:")
            print(classification_report(all_labels, all_preds, target_names=zone_names))

    return metrics



## CHECK FOR PREPROCESSED SEQUENCES AND LOAD IF AVAILABLE



In [ ]:
def check_preprocessed_exists(backend='keras'):
    """Check if all required preprocessed files exist for the specified backend"""

    required_files = [
        f'{PREPROCESSED_DIR}/scaler.pkl',
        f'{PREPROCESSED_DIR}/config.pkl',
    ]

    # Add backend-specific files
    if backend == 'keras':
        required_files.extend([
            f'{PREPROCESSED_DIR}/sequences.h5',
            f'{PREPROCESSED_DIR}/train_df.pkl',
            f'{PREPROCESSED_DIR}/val_df.pkl',
            f'{PREPROCESSED_DIR}/test_df.pkl'
        ])
    else:  # pytorch
        required_files.extend([
            f'{PREPROCESSED_DIR}/train_dataset.pt',
            f'{PREPROCESSED_DIR}/val_dataset.pt',
            f'{PREPROCESSED_DIR}/test_dataset.pt'
        ])

    if not os.path.exists(PREPROCESSED_DIR):
        return False, f"Preprocessed directory not found: {PREPROCESSED_DIR}"

    missing_files = [f for f in required_files if not os.path.exists(f)]

    if missing_files:
        return False, f"Missing files for {backend}: {[os.path.basename(f) for f in missing_files]}"

    return True, f"All {backend} preprocessed files found"


def load_preprocessed_sequences_keras(input_dir=PREPROCESSED_DIR):
    """Load preprocessed sequences for Keras backend"""

    print("\n" + "="*80)
    print("📂 LOADING KERAS PREPROCESSED DATA")
    print("="*80)

    try:
        # 1. Load config first
        config = joblib.load(f'{input_dir}/config.pkl')
        print(f"✓ Loaded config")

        # 2. Load scaler
        scaler = joblib.load(f'{input_dir}/scaler.pkl')
        print(f"✓ Loaded scaler ({scaler.n_features_in_} features)")

        # 3. Load sequences
        print("Loading sequences from HDF5...")
        with h5py.File(f'{input_dir}/sequences.h5', 'r') as f:
            X_train = f['X_train'][:]
            y_train = f['y_train'][:]
            X_val = f['X_val'][:]
            y_val = f['y_val'][:]
            X_test = f['X_test'][:]
            y_test = f['y_test'][:]

        print(f"✓ Loaded sequences:")
        print(f"   Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

        # 4. Load dataframes
        train_df = pd.read_pickle(f'{input_dir}/train_df.pkl')
        val_df = pd.read_pickle(f'{input_dir}/val_df.pkl')
        test_df = pd.read_pickle(f'{input_dir}/test_df.pkl')

        print(f"✓ Loaded dataframes:")
        print(f"   Train: {len(train_df)} rows, Val: {len(val_df)} rows, Test: {len(test_df)} rows")

        # 5. Update global variables
        global SEQ_LEN, HORIZON_FRAMES, HORIZON_SECONDS, N_ROWS, N_COLS
        global NUM_ZONES, FEATURE_COLS, CO_ORDINATES, BATCH_SIZE, BACKEND

        SEQ_LEN = config['seq_len']
        HORIZON_FRAMES = config['horizon_frames']
        HORIZON_SECONDS = config['horizon_seconds']
        N_ROWS = config['n_rows']
        N_COLS = config['n_cols']
        NUM_ZONES = N_ROWS * N_COLS
        FEATURE_COLS = config['feature_cols']
        CO_ORDINATES = config['coordinate_targets']
        BATCH_SIZE = config['batch_size']
        BACKEND = config.get('backend', 'keras')

        print(f"\n✓ Configuration updated:")
        print(f"   SEQ_LEN={SEQ_LEN}, HORIZON={HORIZON_FRAMES} frames ({HORIZON_SECONDS}s)")
        print(f"   Grid: {N_ROWS}x{N_COLS}={NUM_ZONES} zones")
        print(f"   Features: {len(FEATURE_COLS)}")
        print(f"   Coordinate targets: {CO_ORDINATES}")

        print("\n" + "="*80)
        print("✅ KERAS PREPROCESSED DATA LOADED SUCCESSFULLY")
        print("="*80)

        return {
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test,
            'scaler': scaler,
            'train_df': train_df,
            'val_df': val_df,
            'test_df': test_df,
            'df': pd.concat([train_df, val_df, test_df], ignore_index=True)
        }

    except Exception as e:
        print(f"\n❌ Error loading Keras preprocessed data: {e}")
        return None


def load_preprocessed_sequences_pytorch(input_dir=PREPROCESSED_DIR):
    """Load preprocessed sequences for PyTorch backend"""

    print("\n" + "="*80)
    print("📂 LOADING PYTORCH PREPROCESSED DATA")
    print("="*80)

    try:
        # 1. Load config first
        config = joblib.load(f'{input_dir}/config.pkl')
        print(f"✓ Loaded config")

        # 2. Load scaler
        scaler = joblib.load(f'{input_dir}/scaler.pkl')
        print(f"✓ Loaded scaler ({scaler.n_features_in_} features)")

        # 3. Load PyTorch datasets
        print("Loading PyTorch datasets...")
        train_ds = torch.load(f'{input_dir}/train_dataset.pt')
        val_ds = torch.load(f'{input_dir}/val_dataset.pt')
        test_ds = torch.load(f'{input_dir}/test_dataset.pt')

        print(f"✓ Loaded datasets:")
        print(f"   Train: {len(train_ds)} sequences")
        print(f"   Val: {len(val_ds)} sequences")
        print(f"   Test: {len(test_ds)} sequences")

        # 4. Update global variables
        global SEQ_LEN, HORIZON_FRAMES, HORIZON_SECONDS, N_ROWS, N_COLS
        global NUM_ZONES, FEATURE_COLS, CO_ORDINATES, BATCH_SIZE, BACKEND

        SEQ_LEN = config['seq_len']
        HORIZON_FRAMES = config['horizon_frames']
        HORIZON_SECONDS = config['horizon_seconds']
        N_ROWS = config['n_rows']
        N_COLS = config['n_cols']
        NUM_ZONES = N_ROWS * N_COLS
        FEATURE_COLS = config['feature_cols']
        CO_ORDINATES = config['coordinate_targets']
        BATCH_SIZE = config['batch_size']
        BACKEND = config.get('backend', 'torch')

        print(f"\n✓ Configuration updated:")
        print(f"   SEQ_LEN={SEQ_LEN}, HORIZON={HORIZON_FRAMES} frames ({HORIZON_SECONDS}s)")
        print(f"   Grid: {N_ROWS}x{N_COLS}={NUM_ZONES} zones")
        print(f"   Features: {len(FEATURE_COLS)}")
        print(f"   Coordinate targets: {CO_ORDINATES}")

        print("\n" + "="*80)
        print("✅ PYTORCH PREPROCESSED DATA LOADED SUCCESSFULLY")
        print("="*80)

        return {
            'train_ds': train_ds,
            'val_ds': val_ds,
            'test_ds': test_ds,
            'scaler': scaler
        }

    except Exception as e:
        print(f"\n❌ Error loading PyTorch preprocessed data: {e}")
        return None


# ============================================================================
# MAIN DECISION LOGIC
# ============================================================================

print("\n🔍 Checking for preprocessed sequences...")
print(f"Backend: {BACKEND}")

preprocessed_available, message = check_preprocessed_exists(backend=BACKEND)

if preprocessed_available and USE_PREPROCESSED:
    print(f"✅ {message}")
    print(f"📂 Loading from: {PREPROCESSED_DIR}/")

    # Load based on backend
    if BACKEND == "keras":
        loaded_data = load_preprocessed_sequences_keras(PREPROCESSED_DIR)

        if loaded_data is not None:
            # Extract all variables
            X_train = loaded_data['X_train']
            y_train = loaded_data['y_train']
            X_val = loaded_data['X_val']
            y_val = loaded_data['y_val']
            X_test = loaded_data['X_test']
            y_test = loaded_data['y_test']
            scaler = loaded_data['scaler']
            train_df = loaded_data['train_df']
            val_df = loaded_data['val_df']
            test_df = loaded_data['test_df']
            df = loaded_data['df']

            SKIP_DATA_LOADING = True
            print("✅ Keras data loaded into memory")

        else:
            print("⚠️  Failed to load preprocessed Keras data. Will load raw data instead.")
            SKIP_DATA_LOADING = False

    elif BACKEND == "torch":
        loaded_data = load_preprocessed_sequences_pytorch(PREPROCESSED_DIR)

        if loaded_data is not None:
            # Extract PyTorch datasets
            train_ds = loaded_data['train_ds']
            val_ds = loaded_data['val_ds']
            test_ds = loaded_data['test_ds']
            scaler = loaded_data['scaler']

            SKIP_DATA_LOADING = True
            print("✅ PyTorch datasets loaded into memory")

        else:
            print("⚠️  Failed to load preprocessed PyTorch data. Will load raw data instead.")
            SKIP_DATA_LOADING = False

else:
    if not preprocessed_available:
        print(f"❌ {message}")
    else:
        print("⚠️  USE_PREPROCESSED = False, will load raw data")

    print(f"\n📥 Will proceed with normal data loading and preprocessing for {BACKEND}...")
    print("   (This may take 5-10 minutes)")
    SKIP_DATA_LOADING = False



🔍 Checking for preprocessed sequences...
Backend: keras
❌ Preprocessed directory not found: /kaggle/input/preprocessed-sequences/

📥 Will proceed with normal data loading and preprocessing for keras...
   (This may take 5-10 minutes)


## Loading Dataset

In [ ]:
if not SKIP_DATA_LOADING:
    print("Loading dataset...")

    try:
        df = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {df.shape}")

        # Display basic info about the dataset
        print(f"Columns: {list(df.columns)}")
        print(f"Unique matches: {df['match_id'].nunique()}")
        print(f"Unique players: {df['player_id'].nunique()}")

        if 'timestamp_seconds' in df.columns:
            print(f"Time range: {df['timestamp_seconds'].min():.2f} to {df['timestamp_seconds'].max():.2f} seconds")

    except FileNotFoundError:
        print(f"Error: Could not find {file_path}")
        print("Please ensure the combined_matches.csv file exists in the csv_labeled_data folder")
        raise

    # Display all columns
    print("\nAll columns in the DataFrame:")
    for col in df.columns:
        print(f"- {col}")


    # Add enhanced features
    print("Adding velocity and enhanced features...")
    df = add_velocity_features(df)

    # Add contextual features if available
    if hasattr(globals(), 'add_contextual_features'):
        df = add_contextual_features(df)

    # Check for required columns and handle missing values
    print("Checking feature columns...")
    available_features = [col for col in FEATURE_COLS if col in df.columns]
    missing_features = [col for col in FEATURE_COLS if col not in df.columns]

    if missing_features:
        print(f"Warning: Missing columns {missing_features}")
        print(f"Available features: {available_features}")

        # Use only available features
        FEATURE_COLS = available_features
        print(f"Updated feature list to {len(FEATURE_COLS)} available features")

    # Handle missing values
    print("Handling missing values...")
    initial_rows = len(df)
    df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    final_rows = len(df)
    print(f"Removed {initial_rows - final_rows} rows with missing values")

    print(f"Dataset processed successfully! Final shape: {df.shape}")

    # Show data distribution
    print("\nData distribution by match:")
    match_counts = df['match_id'].value_counts().sort_index()
    print(match_counts)

    print("\nSample of processed data:")
    print(df[['match_id', 'player_id', 'frame_number'] + FEATURE_COLS[:5]].head())


else:
    print("✅ Skipping data loading - using preprocessed sequences")
    print(f"   Dataset shape: {df.shape}")
    print(f"   Features already computed: {len(FEATURE_COLS)}")

Loading dataset...
Dataset loaded successfully! Shape: (5330304, 52)
Columns: ['match_id', 'frame_number', 'period', 'timestamp_seconds', 'time_string', 'trackable_object', 'object_type', 'player_id', 'team_id', 'team_name', 'team_type', 'player_name', 'player_number', 'position_raw', 'position_category', 'x', 'y', 'field_zone', 'penalty_area', 'distance_from_center', 'distance_to_ball', 'speed', 'acceleration', 'team_centroid_distance', 'team_spread', 'nearest_opponent_1', 'nearest_opponent_2', 'nearest_opponent_3', 'defensive_line_distance', 'home_team', 'away_team', 'home_score', 'away_score', 'is_goalkeeper', 'is_defender', 'is_midfielder', 'is_forward', 'x_normalized', 'y_normalized', 'speed_normalized', 'ball_possession_proximity', 'defensive_third', 'attacking_third', 'wide_position', 'possession_team', 'possession_player_name', 'possession_player_id', 'possession_position', 'has_possession', 'team_has_possession', 'opponent_has_possession', 'match_number']
Unique matches: 9
Uni

## Data Splitting and Preparation
### fit scaler on training features

In [ ]:
# REPLACE "Data Splitting and Preparation" cell with:


if not SKIP_DATA_LOADING:
    print("Splitting data by matches...")
    train_df, val_df, test_df = split_by_match_df(df, fixed_split=True)


    print("Fitting scaler on training data...")
    scaler = StandardScaler()
    scaler.fit(train_df[FEATURE_COLS].values)
else:
    print("✅ Skipping data splitting - using preprocessed splits")
    print(f"   Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    print(f"   Scaler ready with {scaler.n_features_in_} features")

Splitting data by matches...
Split matches: 4 1 2
Rows per split: 2003172 476723 1607918
Fitting scaler on training data...


## SAVE PREPROCESSED SEQUENCES FOR FAST RELOADING


In [ ]:

def check_memory_usage(threshold=80):
    """Check if memory usage exceeds threshold (%)"""
    memory = psutil.virtual_memory()
    usage_percent = memory.percent
    if usage_percent > threshold:
        print(f"\n⚠️  MEMORY WARNING: {usage_percent:.1f}% used (threshold: {threshold}%)")
        return True
    return False

def save_preprocessed_sequences(output_dir='preprocessed_data', chunk_size=500, memory_threshold=80):
    """Save preprocessed sequences with memory monitoring"""

    if SKIP_DATA_LOADING:
        print("⚠️  Data already loaded from preprocessed files. Skipping save.")
        return

    os.makedirs(output_dir, exist_ok=True)

    print("\n💾 Saving preprocessed sequences (memory-efficient mode)...")
    print(f"Backend: {BACKEND}")
    print(f"Memory threshold: {memory_threshold}%")

    # 1. Save scaler
    joblib.dump(scaler, f'{output_dir}/scaler.pkl')
    print(f"✓ Saved scaler")

    # 2. Save config
    config = {
        'seq_len': SEQ_LEN,
        'horizon_frames': HORIZON_FRAMES,
        'horizon_seconds': HORIZON_SECONDS,
        'n_rows': N_ROWS,
        'n_cols': N_COLS,
        'feature_cols': FEATURE_COLS,
        'coordinate_targets': CO_ORDINATES,
        'batch_size': BATCH_SIZE,
        'backend': BACKEND
    }
    joblib.dump(config, f'{output_dir}/config.pkl')
    print(f"✓ Saved config")

    if BACKEND == "keras":
        print("Saving Keras sequences with memory monitoring...")

        # Save dataframes first
        train_df.to_pickle(f'{output_dir}/train_df.pkl')
        val_df.to_pickle(f'{output_dir}/val_df.pkl')
        test_df.to_pickle(f'{output_dir}/test_df.pkl')
        print(f"✓ Saved dataframe splits")

        with h5py.File(f'{output_dir}/sequences.h5', 'w') as f:
            for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
                print(f"\nProcessing {split_name} split...")

                # Check memory before starting
                if check_memory_usage(memory_threshold):
                    print(f"❌ Memory threshold exceeded before {split_name}. Stopping save.")
                    return False

                gen = keras_sequence_generator(
                    split_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,
                    coordinate_targets=CO_ORDINATES, n_rows=N_ROWS, n_cols=N_COLS
                )

                X_dataset = None
                y_dataset = None
                total_sequences = 0
                batch = []

                for seq, target in gen:
                    # Check memory every chunk
                    if len(batch) > 0 and len(batch) % chunk_size == 0:
                        if check_memory_usage(memory_threshold):
                            print(f"\n❌ Memory threshold exceeded at {total_sequences} sequences for {split_name}")
                            print(f"   Partial data saved. Consider:")
                            print(f"   1. Reducing chunk_size (current: {chunk_size})")
                            print(f"   2. Increasing memory_threshold (current: {memory_threshold}%)")
                            print(f"   3. Using smaller dataset or more RAM")
                            return False

                    batch.append((seq, target))

                    if len(batch) >= chunk_size:
                        X_chunk = np.array([b[0] for b in batch], dtype=np.float32)
                        y_chunk = np.array([b[1] for b in batch], dtype=np.float32 if CO_ORDINATES else np.int32)

                        if X_dataset is None:
                            y_shape = (0, 2) if CO_ORDINATES else (0,)
                            y_maxshape = (None, 2) if CO_ORDINATES else (None,)

                            X_dataset = f.create_dataset(
                                f'X_{split_name}',
                                shape=(0, SEQ_LEN, len(FEATURE_COLS)),
                                maxshape=(None, SEQ_LEN, len(FEATURE_COLS)),
                                dtype=np.float32,
                                compression='gzip',
                                compression_opts=4
                            )

                            y_dataset = f.create_dataset(
                                f'y_{split_name}',
                                shape=y_shape,
                                maxshape=y_maxshape,
                                dtype=np.float32 if CO_ORDINATES else np.int32,
                                compression='gzip',
                                compression_opts=4
                            )

                        X_dataset.resize((total_sequences + len(X_chunk), SEQ_LEN, len(FEATURE_COLS)))
                        X_dataset[total_sequences:total_sequences+len(X_chunk)] = X_chunk

                        y_dataset.resize((total_sequences + len(y_chunk),) + y_chunk.shape[1:])
                        y_dataset[total_sequences:total_sequences+len(y_chunk)] = y_chunk

                        total_sequences += len(X_chunk)

                        mem_percent = psutil.virtual_memory().percent
                        print(f"  {split_name}: {total_sequences:,} sequences | RAM: {mem_percent:.1f}%", end='\r')

                        batch = []
                        del X_chunk, y_chunk
                        import gc
                        gc.collect()

                # Save remaining
                if batch:
                    X_chunk = np.array([b[0] for b in batch], dtype=np.float32)
                    y_chunk = np.array([b[1] for b in batch], dtype=np.float32 if CO_ORDINATES else np.int32)

                    X_dataset.resize((total_sequences + len(X_chunk), SEQ_LEN, len(FEATURE_COLS)))
                    X_dataset[total_sequences:total_sequences+len(X_chunk)] = X_chunk

                    y_dataset.resize((total_sequences + len(y_chunk),) + y_chunk.shape[1:])
                    y_dataset[total_sequences:total_sequences+len(y_chunk)] = y_chunk

                    total_sequences += len(X_chunk)
                    del X_chunk, y_chunk, batch
                    import gc
                    gc.collect()

                print(f"\n✓ Saved {split_name}: {total_sequences:,} sequences")

        print(f"\n✅ All Keras sequences saved successfully")
        return True

    elif BACKEND == "torch":
        print("Saving PyTorch datasets...")

        torch.save(train_ds, f'{output_dir}/train_dataset.pt')
        if check_memory_usage(memory_threshold):
            print(f"⚠️  High memory usage after saving train dataset")

        torch.save(val_ds, f'{output_dir}/val_dataset.pt')
        if check_memory_usage(memory_threshold):
            print(f"⚠️  High memory usage after saving val dataset")

        torch.save(test_ds, f'{output_dir}/test_dataset.pt')

        print(f"✓ Saved PyTorch datasets")
        return True

    print(f"\n✅ All data saved to '{output_dir}/'")
    return True

# Call with memory monitoring
# success = save_preprocessed_sequences(chunk_size=200, memory_threshold=80)
# if not success:
#     print("\n⚠️  Saving stopped due to memory constraints")

## pytorch dataset initialisation into data sequence and loaders

In [ ]:
if BACKEND == "torch":
    if not SKIP_DATA_LOADING:
        print("Creating PyTorch sequence datasets...")

        train_ds = FootballSequenceDataset(
            train_df,
            seq_len=SEQ_LEN,
            horizon=HORIZON_FRAMES,
            feature_cols=FEATURE_COLS,
            scaler=scaler
        )

        val_ds = FootballSequenceDataset(
            val_df,
            seq_len=SEQ_LEN,
            horizon=HORIZON_FRAMES,
            feature_cols=FEATURE_COLS,
            scaler=scaler
        )

        test_ds = FootballSequenceDataset(
            test_df,
            seq_len=SEQ_LEN,
            horizon=HORIZON_FRAMES,
            feature_cols=FEATURE_COLS,
            scaler=scaler
        )

        print(f"Dataset sizes - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
    else:
        print("✅ Using preprocessed PyTorch datasets")
        print(f"   Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

    # Create DataLoaders (always needed)
    print("Creating PyTorch DataLoaders...")

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=worker_num,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        num_workers=worker_num,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        num_workers=worker_num,
        pin_memory=True
    )

    print(f"✓ DataLoaders created with batch_size={BATCH_SIZE}")

## Keras dataset initialisation into data sequence and loaders

In [ ]:

if BACKEND == "keras":
    # Check available GPUs
    gpus = tf.config.list_physical_devices('GPU')
    print(f"\n🔍 GPU Detection:")
    print(f"Available GPUs: {len(gpus)}")
    for gpu in gpus:
        print(f"  - {gpu}")

    # Create distribution strategy for multi-GPU
    if len(gpus) > 1:
        print(f"\n🚀 Using MirroredStrategy for {len(gpus)} GPUs")
        strategy = tf.distribute.MirroredStrategy()
    else:
        print("\n⚠️  Single/No GPU detected, using default strategy")
        strategy = tf.distribute.get_strategy()

    print(f"Number of devices in strategy: {strategy.num_replicas_in_sync}")

    # Adjust batch size for multi-GPU (IMPORTANT!)
    GLOBAL_BATCH_SIZE = BATCH_SIZE * strategy.num_replicas_in_sync
    print(f"\nBatch size configuration:")
    print(f"  • Per-GPU batch size: {BATCH_SIZE}")
    print(f"  • Global batch size: {GLOBAL_BATCH_SIZE}")
else:
    print(f"\n🔥 PyTorch backend - multi-GPU handled differently")
    strategy = None  # Not needed for PyTorch


🔍 GPU Detection:
Available GPUs: 1
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

⚠️  Single/No GPU detected, using default strategy
Number of devices in strategy: 1

Batch size configuration:
  • Per-GPU batch size: 32
  • Global batch size: 32


In [ ]:
if BACKEND == "keras":
       if not SKIP_DATA_LOADING:
        print("Creating Keras sequence datasets via tf.data...")

        train_loader = make_tf_dataset(
            train_df,
            BATCH_SIZE,
            shuffle=True,
            coordinate_targets=CO_ORDINATES,
        )

        val_loader = make_tf_dataset(val_df, BATCH_SIZE, shuffle=False, coordinate_targets=CO_ORDINATES)
        test_loader = make_tf_dataset(test_df, BATCH_SIZE, shuffle=False, coordinate_targets=CO_ORDINATES)

        print(f"Datasets created. Ready for training.")
       else:
            print("✅ Using preprocessed sequences - creating tf.data.Datasets...")

            # Create datasets from arrays
            train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
            train_loader = train_dataset.shuffle(2048).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

            val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
            val_loader = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

            test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
            test_loader = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

            print(f"✓ Train batches: {len(train_loader)}")
            print(f"✓ Val batches: {len(val_loader)}")
            print(f"✓ Test batches: {len(test_loader)}")

Creating Keras sequence datasets via tf.data...
   Dataset: 1,983,336 sequences → 61979 batches
   Dataset: 471,329 sequences → 14729 batches
   Dataset: 1,598,522 sequences → 49953 batches
Datasets created. Ready for training.


In [ ]:
# Reconstruct labels from DataFrames for analysis (since y_train variable doesn't exist)
# Note: This checks the raw potential zones in the DF, not necessarily the sequences (but they should match close enough)
y_train_check = xy_to_zone_vectorized(train_df["x_normalized"], train_df["y_normalized"], N_ROWS, N_COLS)
y_val_check = xy_to_zone_vectorized(val_df["x_normalized"], val_df["y_normalized"], N_ROWS, N_COLS)

print("Train zones range:", np.min(y_train_check), np.max(y_train_check))
print("Val zones range:  ", np.min(y_val_check), np.max(y_val_check))

# Check if Validation has zones that Train never saw
diff = set(np.unique(y_val_check)) - set(np.unique(y_train_check))
print("Zones in Val but NOT in Train:", diff)

# Inspect one batch from the tf.data.Dataset (Keras loader)
print("\nInspecting one batch from train_loader...")
for x_batch, y_batch in train_loader.take(1):
    print("Batch X shape:", x_batch.shape)
    print("Batch X stats (mean/std):", np.mean(x_batch), np.std(x_batch))
    print("Batch Y shape:", y_batch.shape)
    print("Sample Y values:", y_batch.numpy().flatten()[:10])


Train zones range: 0 143
Val zones range:   0 143
Zones in Val but NOT in Train: set()

Inspecting one batch from train_loader...
Batch X shape: (32, 100, 26)
Batch X stats (mean/std): 0.034416154 1.0048648
Batch Y shape: (32,)
Sample Y values: [13  4 28 42 87 68 70 69 50 41]


In [ ]:
import tensorflow as tf
tf.keras.backend.clear_session()
import gc
gc.collect()

0

## Model Initialisation

In [ ]:
# Initialize enhanced model
model_type = MODEL_TYPE
print(f"Initializing {MODEL_TYPE}")

if model_type == "enhanced":
    model = EnhancedFootballModel(
        input_dim=len(FEATURE_COLS),
        num_classes=NUM_ZONES,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3
    ).to(device)

if model_type == "tcn":
    model = TemporalConvNet(
        input_dim=len(FEATURE_COLS),
        num_classes=NUM_ZONES,
        num_channels=[64, 128, 256],
        dropout=0.3
    ).to(device)

if model_type == "tcn_new":
    model = ZoneTCN(input_dim=len(FEATURE_COLS),num_classes=NUM_ZONES,channels=[64, 128, 256, 256, 256, 256],dropout=0.3).to(device)

if model_type == "LocusLab_tcn":
    model = DeltaTCN(
        input_dim=len(FEATURE_COLS),
        num_classes=2,
        channels=[64, 128, 256, 512],
        kernel_size=5,
        dropout=0.1
    ).to(device)

if model_type == "Keras_tcn":
    with strategy.scope():  # ✅ Everything inside scope
        model = compiled_tcn(
            num_feat=len(FEATURE_COLS),
            num_classes = 2 if CO_ORDINATES else NUM_ZONES,
            nb_filters= [64, 128, 128, 128, 256, 256],
            kernel_size=5,
            dilations=[1, 2, 4, 8, 16, 32],
            nb_stacks=2,
            max_len=SEQ_LEN,
            padding='causal',
            use_skip_connections=False,
            return_sequences=False,
            dropout_rate=0.3,
            activation='relu',
            use_batch_norm=True,
            use_layer_norm=False,
            lr=LR,
            regression=CO_ORDINATES
        )

        # ✅ Re-compile INSIDE strategy.scope() with XLA
        if CO_ORDINATES:
            optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5, clipnorm=1.0)
            optimizer = mixed_precision.LossScaleOptimizer(optimizer)  # ⚡ FP16 speedup
            model.compile(
                optimizer=optimizer,
                loss='mse',
                metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')],
                jit_compile=False # 🚀 XLA compilation for speed
            )

        # ✅ Get model info INSIDE scope
        total_params = model.count_params()
        trainable_params = sum([np.prod(w.shape) for w in model.trainable_weights])

        print("Keras model compiled inside strategy.scope()")
        print(f"Using {strategy.num_replicas_in_sync} GPU(s)")

# PyTorch models (unchanged)
if BACKEND == "pytorch":
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for PyTorch model")
        model = nn.DataParallel(model)
    model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Print summary
print(f"Model initialized: {model_type}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Input features: {len(FEATURE_COLS)}")
print(f"Output size: {2 if CO_ORDINATES else NUM_ZONES}")

Initializing Keras_tcn
x.shape= (None, 256)
model.x = (None, 100, 26)
model.y = (None, 144)
Keras model compiled inside strategy.scope()
Using 1 GPU(s)
Model initialized: Keras_tcn
Total parameters: 3,483,152
Trainable parameters: 3,475,472
Input features: 26
Output size: 144


##  MODEL TRAINING

In [ ]:

print("Starting enhanced training...")
print(f"Training configuration:")
print(f"Training for {'regression (coordinates)' if CO_ORDINATES else 'classification (zones)'}")
print(f"- Epochs: {EPOCHS}")
print(f"- Learning rate: {LR}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Patience: {PATIENCE}")
print(f"- Device: {device}")
print(f"-Model Type:{MODEL_TYPE}")
print(f"-Model Type:{BACKEND}")

trained_model, training_history = train_model(model, train_loader, val_loader,epochs=EPOCHS, lr=LR, patience=PATIENCE)

print("Training completed successfully!")




Starting enhanced training...
Training configuration:
Training for classification (zones)
- Epochs: 1
- Learning rate: 0.0005
- Batch size: 32
- Patience: 2
- Device: cuda
-Model Type:Keras_tcn
-Model Type:keras
   4150/Unknown 81s 12ms/step - loss: 3.9645 - sparse_categorical_accuracy: 0.1211 - sparse_top_k_categorical_accuracy: 0.3443

In [ ]:
# Take one batch from validation loader
X_val_batch, y_val_batch = next(iter(val_loader))

# Make predictions
preds = model.predict(X_val_batch, verbose=0)
pred_zones = np.argmax(preds, axis=1)

if hasattr(y_val_batch, 'numpy'):
    true_zones = y_val_batch.numpy().flatten()
else:
    true_zones = y_val_batch.flatten()

# Display comparison
print("="*60)
print("🔍 VALIDATION PREDICTION INSPECTION")
print("="*60)
print(f"True Zones (First 20):      {true_zones[:20]}")
print(f"Predicted Zones (First 20): {pred_zones[:20]}")

# Check if predictions are 'stuck' (predicting same class)
unique_preds = np.unique(pred_zones)
print(f"\nUnique predicted zones in batch: {len(unique_preds)}")
print(f"Most common prediction: {np.bincount(pred_zones).argmax()} (Count: {np.max(np.bincount(pred_zones))}/{len(pred_zones)})")

# Calculate accuracy for this batch
batch_acc = np.mean(pred_zones == true_zones)
print(f"\nBatch Accuracy: {batch_acc:.4f}")
print("="*60)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def debug_data_splits(train_df, val_df, test_df):
    print("="*80)
    print("🔍 DATA SPLIT DEBUGGING")
    print("="*80)

    # 1. Check Match IDs
    print(f"\n1. Match ID Distribution:")
    print(f"   Train Matches: {sorted(train_df['match_id'].unique())}")
    print(f"   Val Matches:   {sorted(val_df['match_id'].unique())}")
    print(f"   Test Matches:  {sorted(test_df['match_id'].unique())}")

    # 2. Check Coordinate Ranges (Crucial for Zone Calculation)
    print(f"\n2. Coordinate Statistics (Normalized):")
    for name, df in [('Train', train_df), ('Val', val_df)]:
        print(f"   {name} - X: min={df['x_normalized'].min():.4f}, max={df['x_normalized'].max():.4f}, mean={df['x_normalized'].mean():.4f}")
        print(f"   {name} - Y: min={df['y_normalized'].min():.4f}, max={df['y_normalized'].max():.4f}, mean={df['y_normalized'].mean():.4f}")

    # 3. Zone Distribution Comparison
    print(f"\n3. Visualizing Zone Distributions...")
    # Calculate zones for visualization check
    train_zones = xy_to_zone_vectorized(train_df['x_normalized'], train_df['y_normalized'], N_ROWS, N_COLS)
    val_zones = xy_to_zone_vectorized(val_df['x_normalized'], val_df['y_normalized'], N_ROWS, N_COLS)

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 2, 1)
    plt.hist(train_zones, bins=NUM_ZONES, alpha=0.7, color='blue', density=True)
    plt.title(f'Train Zone Distribution (N={len(train_zones):,})')
    plt.xlabel('Zone ID')
    plt.ylabel('Density')

    plt.subplot(1, 2, 2)
    plt.hist(val_zones, bins=NUM_ZONES, alpha=0.7, color='red', density=True)
    plt.title(f'Val Zone Distribution (N={len(val_zones):,})')
    plt.xlabel('Zone ID')

    plt.tight_layout()
    plt.show()

    # 4. Check for Empty/NaN features
    print(f"\n4. Missing Value Check:")
    print(f"   Train NaNs: {train_df[FEATURE_COLS].isna().sum().sum()}")
    print(f"   Val NaNs:   {val_df[FEATURE_COLS].isna().sum().sum()}")

# Execute Debugging
if 'train_df' in globals() and 'val_df' in globals():
    debug_data_splits(train_df, val_df, test_df)
else:
    print("⚠️ train_df and val_df not found in memory. Please run the data loading cells first.")

In [ ]:
import os

# Create a specific folder if you want to stay organized
output_dir = '/kaggle/working/'
os.makedirs(output_dir, exist_ok=True)

model_save_path = os.path.join(output_dir, 'Model.keras')

trained_model.save(model_save_path)

print(f"Model saved successfully to: {model_save_path}")

In [ ]:
from tensorflow.keras.models import load_model
import joblib

# You must provide the custom classes in a dictionary
custom_objects = {
    'TCN': TCN,
    'ResidualBlock': ResidualBlock
}

reloaded_model = load_model(model_save_path, custom_objects=custom_objects)
print("Model reloaded successfully!")

scaler_path = os.path.join(output_dir, 'feature_scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to: {scaler_path}")

In [ ]:
# ==== PLOT TRAINING HISTORY ====
# Access the actual dictionary inside the History object
history_dict = training_history.history

plt.figure(figsize=(15, 5))

# --- 1. Training & Validation Loss (regression) ---
plt.subplot(1, 2, 1)
# Use the dictionary to get the values.
# Note: Keras usually uses 'loss' and 'val_loss' (not 'train_loss')
plt.plot(history_dict.get('loss', []), label='Train Loss', color='blue')

val_loss = history_dict.get('val_loss', None)
if val_loss is not None:
    plt.plot(val_loss, label='Val Loss', color='red')

plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)

# --- 2. MAE Metrics (Since you are doing Coordinates) ---
if CO_ORDINATES:
    plt.subplot(1, 2, 2)
    plt.plot(history_dict.get('mae', []), label='Train MAE', color='blue')
    plt.plot(history_dict.get('val_mae', []), label='Val MAE', color='red')
    plt.title('Mean Absolute Error (Pitch Units)')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True)

In [ ]:
import pandas as pd
import numpy as np
import torch
import tensorflow as tf

def inspect_predictions_backend(model, val_loader, backend='pytorch', top_k=3):
    """
    Inspect predictions on first batch of validation data.
    Works for PyTorch and Keras models.
    """
    # Get first batch
    sample_input, sample_target = next(iter(val_loader))

    # --- Handle PyTorch ---
    if backend == 'pytorch':
        model.eval()
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # Targets dict or tensor
        if isinstance(sample_target, dict):
            target_tensor = sample_target['delta'] if CO_ORDINATES else sample_target['zone']
        else:
            target_tensor = sample_target

        sample_input = sample_input.to(device)
        target_tensor = target_tensor.to(device)

        with torch.no_grad():
            preds = model(sample_input)

        if CO_ORDINATES:
            preds_np = preds.cpu().numpy()
            targets_np = target_tensor.cpu().numpy()
            results = pd.DataFrame({
                'True_dx': targets_np[:10, 0],
                'Pred_dx': preds_np[:10, 0],
                'True_dy': targets_np[:10, 1],
                'Pred_dy': preds_np[:10, 1]
            })
            results['Error_X'] = abs(results['True_dx'] - results['Pred_dx'])
            results['Error_Y'] = abs(results['True_dy'] - results['Pred_dy'])
        else:
            preds_class = preds.argmax(dim=1).cpu().numpy()
            targets_np = target_tensor.cpu().numpy()
            results = pd.DataFrame({
                'True_Zone': targets_np[:10],
                'Pred_Zone': preds_class[:10]
            })

    # --- Handle Keras ---
    elif backend == 'keras':
        # Convert tensors to numpy if needed
        if isinstance(sample_input, tf.Tensor):
            sample_input = sample_input.numpy()
        if isinstance(sample_target, tf.Tensor):
            sample_target = sample_target.numpy()
        elif isinstance(sample_target, dict):
            sample_target = list(sample_target.values())[0].numpy()

        preds_np = model(sample_input, training=False).numpy()

        if CO_ORDINATES:
            targets_np = sample_target
            results = pd.DataFrame({
                'True_dx': targets_np[:10, 0],
                'Pred_dx': preds_np[:10, 0],
                'True_dy': targets_np[:10, 1],
                'Pred_dy': preds_np[:10, 1]
            })
            results['Error_X'] = abs(results['True_dx'] - results['Pred_dx'])
            results['Error_Y'] = abs(results['True_dy'] - results['Pred_dy'])
        else:
            preds_class = np.argmax(preds_np, axis=1)
            targets_np = sample_target
            results = pd.DataFrame({
                'True_Zone': targets_np[:10],
                'Pred_Zone': preds_class[:10]
            })

    # Print results
    print("\n--- SANITY CHECK (First 10 Samples) ---")
    print(results.round(4))

    if CO_ORDINATES:
        print("\nPrediction Statistics:")
        print(f"Max Pred Value: {preds_np.max():.4f}")
        print(f"Min Pred Value: {preds_np.min():.4f}")
        print(f"Avg Pred Value: {preds_np.mean():.4f}")
    else:
        if top_k > 1:
            topk_preds = np.argsort(preds_np, axis=1)[:, -top_k:][:10]
            print(f"\nTop-{top_k} predictions for first 10 samples:")
            print(topk_preds)


inspect_predictions_backend(trained_model, val_loader, backend=BACKEND)



In [ ]:
# ==== EVALUATION SECTION ====
print("\n" + "="*80)
print("📊 MODEL EVALUATION ON TEST SET")
print("="*80)

# Ensure output directory exists
output_dir = '/kaggle/working/models'
os.makedirs(output_dir, exist_ok=True)

# Load best model
if BACKEND == "keras":
    try:
        with strategy.scope():
            from tensorflow.keras.models import load_model
            model = load_model(
                f"{output_dir}/best_model_{MODEL_TYPE}.keras",
                custom_objects={'TCN': TCN, 'ResidualBlock': ResidualBlock}
            )
        print(f"✅ Loaded best Keras model")
    except:
        print("⚠️  Using trained model (best model file not found)")
        model = trained_model
else:
    try:
        checkpoint = torch.load(f"{output_dir}/best_model_{MODEL_TYPE}.pth", map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        model.eval()
        print(f"✅ Loaded best PyTorch model")
    except:
        print("⚠️  Using trained model (best model file not found)")
        model = trained_model

# Evaluate model
def evaluate_model_unified(model, test_loader, backend='keras', coordinate_mode=False):
    all_preds = []
    all_labels = []

    if backend == 'keras':
        print("Evaluating Keras model...")
        for batch_X, batch_y in test_loader:
            preds = model.predict(batch_X, verbose=0)
            all_preds.append(preds)
            all_labels.append(batch_y.numpy())

        all_preds = np.concatenate(all_preds, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)

    else:
        print("Evaluating PyTorch model...")
        model.eval()
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                if isinstance(batch_y, dict):
                    batch_y = batch_y['delta'] if coordinate_mode else batch_y['zone']
                preds = model(batch_X)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(batch_y.cpu().numpy() if torch.is_tensor(batch_y) else batch_y)

        all_preds = np.concatenate(all_preds, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)

    if coordinate_mode:
        # Regression metrics for coordinates
        mae = np.mean(np.abs(all_preds - all_labels))
        mse = np.mean((all_preds - all_labels) ** 2)
        rmse = np.sqrt(mse)

        # Per-coordinate metrics
        mae_x = np.mean(np.abs(all_preds[:, 0] - all_labels[:, 0]))
        mae_y = np.mean(np.abs(all_preds[:, 1] - all_labels[:, 1]))

        print("\n📊 Coordinate Prediction Metrics:")
        print(f"  Overall MAE: {mae:.4f}")
        print(f"  Overall RMSE: {rmse:.4f}")
        print(f"  X-coordinate MAE: {mae_x:.4f}")
        print(f"  Y-coordinate MAE: {mae_y:.4f}")

        return {
            'mae': mae,
            'rmse': rmse,
            'mse': mse,
            'mae_x': mae_x,
            'mae_y': mae_y,
            'predictions': all_preds,
            'labels': all_labels
        }
    else:
        # Classification metrics for zones
        pred_classes = np.argmax(all_preds, axis=1)
        true_classes = np.argmax(all_labels, axis=1) if all_labels.ndim > 1 else all_labels

        accuracy = accuracy_score(true_classes, pred_classes)

        print("\n📊 Classification Metrics:")
        print(f"  Test Accuracy: {accuracy:.4f}")
        print("\nClassification Report:")
        print(classification_report(true_classes, pred_classes, zero_division=0))

        return {
            'accuracy': accuracy,
            'predictions': pred_classes,
            'labels': true_classes,
            'probabilities': all_preds
        }

# Run evaluation
results = evaluate_model_unified(model, test_loader, backend=BACKEND, coordinate_mode=CO_ORDINATES)

# Save predictions
if CO_ORDINATES:
    print(f"\n💾 Saving coordinate predictions...")
    predictions_df = pd.DataFrame({
        'true_x': results['labels'][:, 0],
        'true_y': results['labels'][:, 1],
        'pred_x': results['predictions'][:, 0],
        'pred_y': results['predictions'][:, 1],
        'error_x': np.abs(results['predictions'][:, 0] - results['labels'][:, 0]),
        'error_y': np.abs(results['predictions'][:, 1] - results['labels'][:, 1])
    })
else:
    print(f"\n💾 Saving zone predictions...")
    predictions_df = pd.DataFrame({
        'true_zone': results['labels'],
        'pred_zone': results['predictions'],
        'correct': results['labels'] == results['predictions']
    })

predictions_df.to_csv(f'{output_dir}/test_predictions.csv', index=False)
print(f"✅ Predictions saved to {output_dir}/test_predictions.csv")

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)

In [ ]:
# ==== VISUALIZATION OF RESULTS ====
if CO_ORDINATES:
    print("\n📊 Visualizing Coordinate Predictions...")

    # Sample for visualization
    sample_size = min(100, len(results['predictions']))
    sample_indices = np.random.choice(len(results['predictions']), sample_size, replace=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Scatter plot: Predicted vs True
    axes[0].scatter(results['labels'][sample_indices, 0], results['predictions'][sample_indices, 0],
                   alpha=0.5, label='X-coordinate', s=30)
    axes[0].scatter(results['labels'][sample_indices, 1], results['predictions'][sample_indices, 1],
                   alpha=0.5, label='Y-coordinate', s=30)
    axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect prediction', linewidth=2)
    axes[0].set_xlabel('True Values', fontsize=12)
    axes[0].set_ylabel('Predicted Values', fontsize=12)
    axes[0].set_title('Predicted vs True Coordinates', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Error distribution
    errors = np.sqrt((results['predictions'][:, 0] - results['labels'][:, 0])**2 +
                     (results['predictions'][:, 1] - results['labels'][:, 1])**2)
    axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[1].set_xlabel('Euclidean Distance Error', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title(f'Prediction Error Distribution\nMean Error: {errors.mean():.4f}',
                     fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f'{output_dir}/prediction_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"✅ Visualization saved to {output_dir}/prediction_visualization.png")
else:
    # Confusion matrix for zone classification
    from sklearn.metrics import confusion_matrix
    import seaborn as sns

    print("\n📊 Generating Confusion Matrix...")
    cm = confusion_matrix(results['labels'], results['predictions'])

    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix - Zone Prediction', fontsize=16, fontweight='bold')
    plt.ylabel('True Zone', fontsize=12)
    plt.xlabel('Predicted Zone', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"✅ Confusion matrix saved to {output_dir}/confusion_matrix.png")

In [ ]:
# ==== FINAL SUMMARY ====
print("\n" + "="*80)
print("🏆 TRAINING & EVALUATION SUMMARY")
print("="*80)

print("\n📊 Dataset Information:")
print(f"  • Total sequences: {len(train_df) + len(val_df) + len(test_df):,}")
print(f"  • Training: {len(train_df):,}")
print(f"  • Validation: {len(val_df):,}")
print(f"  • Test: {len(test_df):,}")

print("\n🧠 Model Configuration:")
print(f"  • Model Type: {MODEL_TYPE}")
print(f"  • Backend: {BACKEND}")
print(f"  • Sequence Length: {SEQ_LEN} frames")
print(f"  • Horizon: {HORIZON_SECONDS}s ({HORIZON_FRAMES} frames)")
print(f"  • Features: {len(FEATURE_COLS)}")
print(f"  • Prediction Mode: {'Coordinates (Regression)' if CO_ORDINATES else 'Zones (Classification)'}")

print("\n📈 Training Configuration:")
print(f"  • Batch Size: {BATCH_SIZE}")
if BACKEND == 'keras':
    print(f"  • Global Batch Size (multi-GPU): {GLOBAL_BATCH_SIZE}")
print(f"  • Learning Rate: {LR}")
if BACKEND == 'keras':
    print(f"  • Epochs Completed: {len(training_history.history.get('loss', []))}")
else:
    print(f"  • Epochs Completed: {len(training_history.get('train_loss', []))}")

print("\n🎯 Final Performance:")
if CO_ORDINATES:
    print(f"  • Test MAE: {results['mae']:.4f}")
    print(f"  • Test RMSE: {results['rmse']:.4f}")
    print(f"  • X-coordinate MAE: {results['mae_x']:.4f}")
    print(f"  • Y-coordinate MAE: {results['mae_y']:.4f}")
else:
    print(f"  • Test Accuracy: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")

print("\n💾 Saved Files:")
print(f"  • Model: {output_dir}/best_model_{MODEL_TYPE}.keras")
print(f"  • Predictions: {output_dir}/test_predictions.csv")
if CO_ORDINATES:
    print(f"  • Visualization: {output_dir}/prediction_visualization.png")
else:
    print(f"  • Confusion Matrix: {output_dir}/confusion_matrix.png")

print("\n" + "="*80)
print("✅ ALL PROCESSING COMPLETE!")
print("="*80)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

def get_player_features(df, player_id):
    """
    Extracts the features for a single player from the dataset.

    Args:
        df: pandas DataFrame containing match data
        player_id: id of the player to select

    Returns:
        torch.Tensor of shape [num_features]
    """
    # Filter the player's latest available row
    player_row = df[df["player_id"] == player_id].iloc[-1]  # latest row

    # Extract the features as a tensor
    features = torch.tensor(player_row[FEATURE_COLS].values, dtype=torch.float32)
    return features


def get_player_sequence_features(df, match_id, player_id, start_frame=None, seq_len=SEQ_LEN):
    """
    Extracts a sequence of features for a single player from the dataset.

    Args:
        df: pandas DataFrame containing match data
        match_id: id of the match
        player_id: id of the player to select
        start_frame: frame to start the sequence from (if None, uses latest available)
        seq_len: length of the sequence

    Returns:
        torch.Tensor of shape [seq_len, num_features] or None if insufficient data
    """
    # Filter the player's data for the specific match
    player_data = df[(df["match_id"] == match_id) & (df["player_id"] == player_id)].copy()

    if len(player_data) < seq_len:
        print(f"Insufficient data for player {player_id} in match {match_id}. "
              f"Need {seq_len} frames, have {len(player_data)}")
        return None

    # Sort by frame number
    player_data = player_data.sort_values("frame_number").reset_index(drop=True)

    # Select the sequence
    if start_frame is None:
        # Use the most recent sequence
        start_idx = len(player_data) - seq_len
    else:
        # Find the index for the specified frame
        frame_indices = player_data[player_data["frame_number"] >= start_frame].index
        if len(frame_indices) < seq_len:
            print(f"Not enough frames after frame {start_frame}")
            return None
        start_idx = frame_indices[0]

    end_idx = start_idx + seq_len
    sequence_data = player_data.iloc[start_idx:end_idx]

    # Check if we have all required features
    available_features = [col for col in FEATURE_COLS if col in sequence_data.columns]
    if len(available_features) < len(FEATURE_COLS):
        missing = set(FEATURE_COLS) - set(available_features)
        print(f"Warning: Missing features {missing}. Using available features: {available_features}")

    # Extract the features as a tensor
    features = torch.tensor(sequence_data[available_features].values, dtype=torch.float32)

    return features


def predict_next_zones_for_player(model, df, match_id, player_id, scaler=None,
                                 start_frame=None, top_k=5, return_all_probs=False):
    """
    Predicts the next-zone probabilities for a selected player using sequence data.

    Args:
        model: trained PyTorch model
        df: pandas DataFrame with player features
        match_id: id of the match
        player_id: id of the player to predict for
        scaler: fitted StandardScaler for feature normalization
        start_frame: frame to start prediction from (None for latest)
        top_k: number of top probable zones to return
        return_all_probs: whether to return probabilities for all zones

    Returns:
        dict with prediction results
    """
    model.eval()

    # Get player sequence
    player_sequence = get_player_sequence_features(df, match_id, player_id, start_frame)

    if player_sequence is None:
        return None

    # Apply scaling if provided
    if scaler is not None:
        seq_len, num_features = player_sequence.shape
        player_sequence_flat = player_sequence.numpy().reshape(-1, num_features)
        player_sequence_scaled = scaler.transform(player_sequence_flat)
        player_sequence = torch.tensor(player_sequence_scaled.reshape(seq_len, num_features),
                                     dtype=torch.float32)

    # Add batch dimension and move to device
    player_sequence = player_sequence.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(player_sequence)
        probs = F.softmax(logits, dim=1).squeeze(0)
        top_probs, top_indices = torch.topk(probs, k=min(top_k, len(probs)))

    # Convert zones to grid coordinates
    def zone_to_coordinates(zone_idx):
        row = zone_idx // N_COLS
        col = zone_idx % N_COLS
        return row, col

    # Prepare results
    results = {
        'player_id': player_id,
        'match_id': match_id,
        'top_zones': top_indices.cpu().numpy(),
        'top_probabilities': top_probs.cpu().numpy(),
        'top_coordinates': [zone_to_coordinates(idx.item()) for idx in top_indices]
    }

    if return_all_probs:
        results['all_probabilities'] = probs.cpu().numpy()

    return results


def visualize_zone_predictions(prediction_results, title=None):
    """
    Visualize the predicted zones on a football field grid.

    Args:
        prediction_results: results from predict_next_zones_for_player
        title: optional title for the plot
    """
    if prediction_results is None:
        print("No prediction results to visualize")
        return

    # Create probability grid
    prob_grid = np.zeros((N_ROWS, N_COLS))

    # Fill in the top predicted zones
    for zone_idx, prob in zip(prediction_results['top_zones'],
                             prediction_results['top_probabilities']):
        row, col = zone_idx // N_COLS, zone_idx % N_COLS
        prob_grid[row, col] = prob

    # Create the plot
    plt.figure(figsize=(12, 6))
    sns.heatmap(prob_grid, annot=True, fmt='.3f', cmap='YlOrRd',
                cbar_kws={'label': 'Probability'})

    if title is None:
        title = f"Zone Predictions for Player {prediction_results['player_id']} in Match {prediction_results['match_id']}"

    plt.title(title)
    plt.xlabel('Field Width (Zones)')
    plt.ylabel('Field Length (Zones)')
    plt.tight_layout()
    plt.show()

    # Print top predictions
    print(f"\nTop {len(prediction_results['top_zones'])} predicted zones:")
    for i, (zone, prob, coords) in enumerate(zip(prediction_results['top_zones'],
                                                prediction_results['top_probabilities'],
                                                prediction_results['top_coordinates'])):
        print(f"{i+1}. Zone {zone} (Row {coords[0]}, Col {coords[1]}): {prob:.4f}")


def predict_for_multiple_players(model, df, match_id, player_ids, scaler=None, top_k=3):
    """
    Predict zones for multiple players in a match.

    Args:
        model: trained model
        df: dataframe with player data
        match_id: match to analyze
        player_ids: list of player IDs to predict for
        scaler: fitted scaler
        top_k: number of top zones to return per player

    Returns:
        dict with results for each player
    """
    results = {}

    for player_id in player_ids:
        print(f"Predicting for player {player_id}...")
        prediction = predict_next_zones_for_player(
            model, df, match_id, player_id, scaler, top_k=top_k
        )

        if prediction is not None:
            results[player_id] = prediction
            print(f"Top prediction: Zone {prediction['top_zones'][0]} "
                  f"(probability: {prediction['top_probabilities'][0]:.4f})")
        else:
            print(f"Could not generate prediction for player {player_id}")

    return results


In [ ]:
# Replace the prediction cell with this fixed version:

# ============================================================================
# ZONE PREDICTION EXAMPLES AND DEMONSTRATIONS
# ============================================================================
# This cell demonstrates how to make predictions using the trained model
# ============================================================================
output_dir='/kaggle/working/'

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import json

# ⚠️ DO NOT REDEFINE FEATURE_COLS HERE - Use the one from training!
# The FEATURE_COLS variable is already defined in earlier cells during training

print(f"Using {len(FEATURE_COLS)} features for prediction (same as training)")
print(f"Features: {FEATURE_COLS}")

def get_player_sequence_features(df, match_id, player_id, start_frame=None, seq_len=SEQ_LEN):
    """
    Extracts a sequence of features for a single player from the dataset.

    Args:
        df: pandas DataFrame containing match data
        match_id: id of the match
        player_id: id of the player to select
        start_frame: frame to start the sequence from (if None, uses latest available)
        seq_len: length of the sequence

    Returns:
        torch.Tensor of shape [seq_len, num_features] or None if insufficient data
    """
    # Filter the player's data for the specific match
    player_data = df[(df["match_id"] == match_id) & (df["player_id"] == player_id)].copy()

    if len(player_data) < seq_len:
        print(f"Insufficient data for player {player_id} in match {match_id}. "
              f"Need {seq_len} frames, have {len(player_data)}")
        return None

    # Sort by frame number
    player_data = player_data.sort_values("frame_number").reset_index(drop=True)

    # Select the sequence
    if start_frame is None:
        # Use the most recent sequence
        start_idx = len(player_data) - seq_len
    else:
        # Find the index for the specified frame
        frame_indices = player_data[player_data["frame_number"] >= start_frame].index
        if len(frame_indices) < seq_len:
            print(f"Not enough frames after frame {start_frame}")
            return None
        start_idx = frame_indices[0]

    end_idx = start_idx + seq_len
    sequence_data = player_data.iloc[start_idx:end_idx]

    # Check if we have all required features
    available_features = [col for col in FEATURE_COLS if col in sequence_data.columns]
    if len(available_features) < len(FEATURE_COLS):
        missing = set(FEATURE_COLS) - set(available_features)
        print(f"Warning: Missing features {missing}. Using available features: {available_features}")
        # Use only available features
        features_to_use = available_features
    else:
        features_to_use = FEATURE_COLS

    # Extract the features as a tensor
    features = torch.tensor(sequence_data[features_to_use].values, dtype=torch.float32)

    return features


def predict_next_zones_for_player(model, df, match_id, player_id, scaler=None,
                                 start_frame=None, top_k=5, return_all_probs=False):
    """
    Predicts the next-zone probabilities for a selected player using sequence data.

    Args:
        model: trained PyTorch model
        df: pandas DataFrame with player features
        match_id: id of the match
        player_id: id of the player to predict for
        scaler: fitted StandardScaler for feature normalization
        start_frame: frame to start prediction from (None for latest)
        top_k: number of top probable zones to return
        return_all_probs: whether to return probabilities for all zones

    Returns:
        dict with prediction results
    """
    model.eval()

    # Get player sequence
    player_sequence = get_player_sequence_features(df, match_id, player_id, start_frame)

    if player_sequence is None:
        return None

    # Apply scaling if provided
    if scaler is not None:
        seq_len, num_features = player_sequence.shape

        # ✅ FIXED: Check if number of features matches scaler
        if num_features != scaler.n_features_in_:
            print(f"⚠️  Feature mismatch detected!")
            print(f"   Sequence has {num_features} features")
            print(f"   Scaler expects {scaler.n_features_in_} features")
            print(f"   This may cause prediction errors.")
            return None

        player_sequence_flat = player_sequence.numpy().reshape(-1, num_features)
        player_sequence_scaled = scaler.transform(player_sequence_flat)
        player_sequence = torch.tensor(player_sequence_scaled.reshape(seq_len, num_features),
                                     dtype=torch.float32)

    # Add batch dimension and move to device
    player_sequence = player_sequence.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(player_sequence)
        probs = F.softmax(logits, dim=1).squeeze(0)
        top_probs, top_indices = torch.topk(probs, k=min(top_k, len(probs)))

    # Convert zones to grid coordinates
    def zone_to_coordinates(zone_idx):
        row = zone_idx // N_COLS
        col = zone_idx % N_COLS
        return row, col

    # Prepare results
    results = {
        'player_id': player_id,
        'match_id': match_id,
        'top_zones': top_indices.cpu().numpy(),
        'top_probabilities': top_probs.cpu().numpy(),
        'top_coordinates': [zone_to_coordinates(idx.item()) for idx in top_indices]
    }

    if return_all_probs:
        results['all_probabilities'] = probs.cpu().numpy()

    return results


def visualize_zone_predictions(prediction_results, title=None):
    """
    Visualize the predicted zones on a football field grid.

    Args:
        prediction_results: results from predict_next_zones_for_player
        title: optional title for the plot
    """
    if prediction_results is None:
        print("No prediction results to visualize")
        return

    # Create probability grid
    prob_grid = np.zeros((N_ROWS, N_COLS))

    # Fill in the top predicted zones
    for zone_idx, prob in zip(prediction_results['top_zones'],
                             prediction_results['top_probabilities']):
        row, col = zone_idx // N_COLS, zone_idx % N_COLS
        prob_grid[row, col] = prob

    # Create the plot
    plt.figure(figsize=(12, 6))
    sns.heatmap(prob_grid, annot=True, fmt='.3f', cmap='YlOrRd',
                cbar_kws={'label': 'Probability'})

    if title is None:
        title = f"Zone Predictions for Player {prediction_results['player_id']} in Match {prediction_results['match_id']}"

    plt.title(title)
    plt.xlabel('Field Width (Zones)')
    plt.ylabel('Field Length (Zones)')
    plt.tight_layout()
    plt.show()

    # Print top predictions
    print(f"\nTop {len(prediction_results['top_zones'])} predicted zones:")
    for i, (zone, prob, coords) in enumerate(zip(prediction_results['top_zones'],
                                                prediction_results['top_probabilities'],
                                                prediction_results['top_coordinates'])):
        print(f"{i+1}. Zone {zone} (Row {coords[0]}, Col {coords[1]}): {prob:.4f}")


def predict_for_multiple_players(model, df, match_id, player_ids, scaler=None, top_k=3):
    """
    Predict zones for multiple players in a match.

    Args:
        model: trained model
        df: dataframe with player data
        match_id: match to analyze
        player_ids: list of player IDs to predict for
        scaler: fitted scaler
        top_k: number of top zones to return per player

    Returns:
        dict with results for each player
    """
    results = {}

    for player_id in player_ids:
        print(f"Predicting for player {player_id}...")
        prediction = predict_next_zones_for_player(
            model, df, match_id, player_id, scaler, top_k=top_k
        )

        if prediction is not None:
            results[player_id] = prediction
            print(f"Top prediction: Zone {prediction['top_zones'][0]} "
                  f"(probability: {prediction['top_probabilities'][0]:.4f})")
        else:
            print(f"Could not generate prediction for player {player_id}")

    return results


# ============================================================================
# PREDICTION EXAMPLES START HERE
# ============================================================================

print("\n" + "="*80)
print("🎯 ZONE PREDICTION DEMONSTRATIONS")
print("="*80)

# Get available matches and players for examples
available_matches = df['match_id'].unique()
print(f"\nAvailable matches: {available_matches}")

# Select a match for demonstration
demo_match = available_matches[0]
demo_players = df[df['match_id'] == demo_match]['player_id'].unique()[:5]  # First 5 players

print(f"Using match {demo_match} for demonstration")
print(f"Demo players: {demo_players}")

# ============================================================================
# Example 1: Single player prediction
# ============================================================================
print("\n" + "="*80)
print("EXAMPLE 1: Single Player Zone Prediction")
print("="*80)

demo_player = demo_players[0]
prediction_result = predict_next_zones_for_player(
    trained_model, df, demo_match, demo_player,
    scaler=scaler, top_k=5, return_all_probs=False
)

if prediction_result:
    print(f"\n✅ Prediction for Player {demo_player} in Match {demo_match}:")
    print(f"Top 5 predicted zones:")
    for i, (zone, prob, coords) in enumerate(zip(
        prediction_result['top_zones'],
        prediction_result['top_probabilities'],
        prediction_result['top_coordinates']
    )):
        print(f"  {i+1}. Zone {zone} (Row {coords[0]}, Col {coords[1]}): {prob:.4f}")

    # Visualize the prediction
    visualize_zone_predictions(prediction_result)
else:
    print(f"❌ Could not generate prediction for player {demo_player}")

# ============================================================================
# Example 2: Multiple players prediction
# ============================================================================
print("\n" + "="*80)
print("EXAMPLE 2: Multiple Players Zone Prediction")
print("="*80)

multi_results = predict_for_multiple_players(
    trained_model, df, demo_match, demo_players[:3],
    scaler=scaler, top_k=3
)

if multi_results:
    print(f"\n✅ Summary for top 3 players in match {demo_match}:")
    for player_id, result in multi_results.items():
        top_zone = result['top_zones'][0]
        top_prob = result['top_probabilities'][0]
        top_coords = result['top_coordinates'][0]
        print(f"Player {player_id}: Zone {top_zone} (Row {top_coords[0]}, Col {top_coords[1]}) - {top_prob:.4f}")

# ============================================================================
# Example 3: Real-time prediction function
# ============================================================================
print("\n" + "="*80)
print("EXAMPLE 3: Real-time Prediction Function")
print("="*80)

def predict_player_next_zone(match_id, player_id, top_k=3):
    """
    Convenient function for real-time predictions
    """
    result = predict_next_zones_for_player(
        trained_model, df, match_id, player_id,
        scaler=scaler, top_k=top_k
    )

    if result:
        print(f"Player {player_id} in Match {match_id}:")
        for i, (zone, prob, coords) in enumerate(zip(
            result['top_zones'],
            result['top_probabilities'],
            result['top_coordinates']
        )):
            print(f"  {i+1}. Zone {zone} (Row {coords[0]}, Col {coords[1]}): {prob:.3f}")
        return result
    else:
        print(f"No prediction available for Player {player_id}")
        return None

# Test the convenience function
print("\nTesting real-time prediction function:")
test_result = predict_player_next_zone(demo_match, demo_players[1])

print("\n" + "="*80)
print("✅ Zone prediction system ready!")
print("Use predict_player_next_zone(match_id, player_id) for quick predictions")
print("="*80)

# ============================================================================
# SAVE PREDICTION EXAMPLES
# ============================================================================
print("\n💾 Saving prediction examples...")

# Save single player prediction results
if prediction_result:
    prediction_data = {
        'player_id': str(prediction_result['player_id']),
        'match_id': str(prediction_result['match_id']),
        'top_zones': prediction_result['top_zones'].tolist(),
        'top_probabilities': prediction_result['top_probabilities'].tolist(),
        'top_coordinates': prediction_result['top_coordinates']
    }
    with open(f'{output_dir}/example_prediction_single.json', 'w') as f:
        json.dump(prediction_data, f, indent=2)
    print(f"✓ Saved single player prediction: {output_dir}/example_prediction_single.json")

# Save multiple player predictions
if multi_results:
    multi_predictions = {}
    for player_id, result in multi_results.items():
        multi_predictions[str(player_id)] = {
            'match_id': str(result['match_id']),
            'top_zones': result['top_zones'].tolist(),
            'top_probabilities': result['top_probabilities'].tolist(),
            'top_coordinates': result['top_coordinates']
        }
    with open(f'{output_dir}/example_predictions_multiple.json', 'w') as f:
        json.dump(multi_predictions, f, indent=2)
    print(f"✓ Saved multiple player predictions: {output_dir}/example_predictions_multiple.json")

# Save prediction visualization
if prediction_result:
    plt.figure(figsize=(12, 6))
    prob_grid = np.zeros((N_ROWS, N_COLS))
    for zone_idx, prob in zip(prediction_result['top_zones'], prediction_result['top_probabilities']):
        row, col = zone_idx // N_COLS, zone_idx % N_COLS
        prob_grid[row, col] = prob
    sns.heatmap(prob_grid, annot=True, fmt='.3f', cmap='YlOrRd', cbar_kws={'label': 'Probability'})
    plt.title(f"Zone Predictions - Player {prediction_result['player_id']}")
    plt.xlabel('Field Width (Zones)')
    plt.ylabel('Field Length (Zones)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/prediction_heatmap.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved prediction heatmap: {output_dir}/prediction_heatmap.png")
    plt.close()

print("\n✅ Prediction examples completed!")

In [ ]:
# ==== MODEL SAVING AND SYSTEM SUMMARY ====
import pickle
from datetime import datetime

# Save the complete model and components
print("Saving trained model and components...")

# Save model state
torch.save({
    'model_state_dict': trained_model.state_dict(),
    'model_config': {
        'input_dim': len(FEATURE_COLS),
        'num_classes': NUM_ZONES,
        'hidden_dim': 128,
        'num_layers': 2,
        'dropout': 0.3
    },
    'training_config': {
        'seq_len': SEQ_LEN,
        'horizon_frames': HORIZON_FRAMES,
        'horizon_seconds': HORIZON_SECONDS,
        'n_rows': N_ROWS,
        'n_cols': N_COLS,
        'feature_cols': FEATURE_COLS,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'lr': LR
    },
    'performance': {
        'test_accuracy': test_acc,
        'test_top3_accuracy': test_top3_acc,
        'training_history': training_history
    }
}, 'football_zone_prediction_model.pth')

# Save the scaler
with open('football_model_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model and scaler saved successfully!")

# Create a comprehensive summary
print("\n" + "="*80)
print("FOOTBALL ZONE PREDICTION SYSTEM - COMPLETE SUMMARY")
print("="*80)

print("📊 DATASET INFORMATION:")
print(f"   • Total records: {len(df):,}")
print(f"   • Unique matches: {df['match_id'].nunique()}")
print(f"   • Unique players: {df['player_id'].nunique()}")
print(f"   • Training sequences: {len(train_ds):,}")
print(f"   • Validation sequences: {len(val_ds):,}")
print(f"   • Test sequences: {len(test_ds):,}")

print("\n🏟️ FIELD CONFIGURATION:")
print(f"   • Grid size: {N_ROWS} x {N_COLS} = {NUM_ZONES} zones")
print(f"   • Prediction horizon: {HORIZON_SECONDS} seconds ({HORIZON_FRAMES} frames)")
print(f"   • Sequence length: {SEQ_LEN} frames")

print("\n🧠 MODEL ARCHITECTURE:")
print(f"   • Type: Enhanced LSTM-CNN with Attention")
print(f"   • Input features: {len(FEATURE_COLS)}")
print(f"   • Hidden dimensions: 128")
print(f"   • LSTM layers: 2 (bidirectional)")
print(f"   • Total parameters: {sum(p.numel() for p in trained_model.parameters()):,}")

print("\n📈 PERFORMANCE METRICS:")
print(f"   • Test Accuracy (Top-1): {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   • Test Accuracy (Top-3): {test_top3_acc:.4f} ({test_top3_acc*100:.2f}%)")
print(f"   • Training epochs completed: {len(training_history['train_acc'])}")

print("\n🎯 FEATURE SET:")
for i, feature in enumerate(FEATURE_COLS, 1):
    print(f"   {i:2d}. {feature}")

print("\n💾 SAVED FILES:")
print("   • football_zone_prediction_model.pth - Complete model checkpoint")
print("   • football_model_scaler.pkl - Feature scaler")
print("   • best_football_model.pth - Best model weights")

print("\n🚀 USAGE:")
print("   Use predict_player_next_zone(match_id, player_id) for quick predictions")
print("   Use predict_next_zones_for_player() for detailed analysis")
print("   Use visualize_zone_predictions() for visual results")

print("\n⚽ SYSTEM CAPABILITIES:")
print("   ✓ Predicts player movement zones 5 seconds ahead")
print("   ✓ Provides probability distributions across all field zones")
print("   ✓ Handles multiple players and matches")
print("   ✓ Includes visual feedback and detailed analytics")
print("   ✓ Uses temporal sequences for context-aware predictions")

current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"\n🕒 System completed at: {current_time}")
print("="*80)

print("\n✅ Football Zone Prediction System is ready for use!")

In [ ]:
# ==== FINAL EVALUATION ====
print("Evaluating model on test set...")

# Detailed evaluation
test_acc, test_top3_acc, test_probs = evaluate_with_metrics(trained_model, test_loader)

print(f"Final Test Results:")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Top-3 Accuracy: {test_top3_acc:.4f}")

# Generate zone names for detailed reporting
zone_names = [f"Zone_{i}" for i in range(NUM_ZONES)]

# Detailed classification report (on a subset to avoid overwhelming output)
print("\nGenerating detailed classification report...")
detailed_acc, test_preds, test_labels = evaluate_detailed(trained_model, test_loader, zone_names)

# Performance summary
print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"Grid Configuration: {N_ROWS} x {N_COLS} = {NUM_ZONES} zones")
print(f"Prediction Horizon: {HORIZON_SECONDS} seconds ({HORIZON_FRAMES} frames)")
print(f"Features Used: {len(FEATURE_COLS)}")
print(f"Model Type: Enhanced LSTM-CNN Architecture")
print(f"Total Parameters: {sum(p.numel() for p in trained_model.parameters()):,}")
print("\nAccuracy Metrics:")
print(f"- Top-1 Accuracy: {test_acc:.4f}")
print(f"- Top-3 Accuracy: {test_top3_acc:.4f}")
print("="*60)